# P04 · Notebook 27 · ToyCar and ToyConveyor: the kept system on two machine types that no notebook has read

**Where we stand.** The kept system is the frozen recipe (two fine-tuned BEATs encoders with ECHO) fused with a second view (TASTgram-MFN trained with Noisy-ArcMix) by the calibrated sum ZP. On DCASE 2020's four MIMII types its evaluation AUC / pAUC is: fan 0.9980 / 0.9894, pump 0.9812 / 0.9202, slide rail 0.9992 / 0.9960, valve 0.9919 / 0.9652 (notebook 25). Every number of the program rests on recordings that were read more than once.

**What it does:** the challenge's two other machine types, ToyCar and ToyConveyor, which no notebook of this project has opened.
1. **Checks first.** The BEATs code, both BEATs files and ECHO's weights must be notebook 23's. Then the six zips are downloaded (MD5 against Zenodo) and every clip is checked.
2. **Part 1 · the frozen recipe,** with notebook 21's settings unchanged: six trainings of 10 epochs on these types' own lists (13 classes), and ECHO on every clip.
3. **Part 2 · the second view,** with notebook 25's settings unchanged: three trainings of 300 epochs.
4. **The development set** is read for the first time. It selects nothing: no form, weight, seed or epoch is chosen on these types. A gate (G-DEV) stops the run only if a system looks broken.
5. **The evaluation set is scored once.** Three sets of scores are written before any label is read: the kept system's, which is the result, and each of its two parts'. The official evaluator is then run on each.
6. **The six types together,** beside the program's five teams.

**Rules:** `results/mimii_beats/RULES_toy_types_written_before_running.md`, written before this notebook existed.

**It needs** in Google Drive: `P04_BEATs`, with the BEATs checkpoint that notebook 1 left there. The BEATs iter3 file is read from the same folder; if it is not there it is downloaded again. It writes to a new folder, `P04_ToyTypes` (2 GB is enough).

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- About 6.4 GB of downloads from Zenodo, six short trainings and three long ones.
- **Time:** about 1.5–2 hours on G4, from the step times notebooks 23 and 26 measured there. A T4 would take many times longer. Cells 10 and 14 print their own estimates for this GPU.
- Everything needed is saved to Drive. If Colab disconnects, run all again: finished trainings are skipped, an unfinished one continues from its last save, and the audio is downloaded again only if the local disk was reset.
- Once the evaluation set has been scored, a later run reads the saved results and scores nothing again.

**At the end** `p04_n27_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, base64, importlib
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_ToyTypes", Path("/content/work"), MY / "P04_BEATs"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_ToyTypes")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", "P04_BEATs"))
if not CKPT_DIR.exists():
    raise SystemExit(f"missing {CKPT_DIR}: the folder P04_BEATs (with BEATs_iter3_plus_AS2M.pt, as notebook 1 left it) must be in Google Drive › MyDrive")
for p in (OUT, WORK):
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_toy_types_written_before_running.md"
# DCASE 2020 Task 2 on Zenodo: development, additional training, evaluation and ground truth (MD5s read from the record pages)
ZD, ZX, ZE, ZG = ("https://zenodo.org/records/3678171/files/", "https://zenodo.org/records/3727685/files/",
                  "https://zenodo.org/records/3841772/files/", "https://zenodo.org/records/3951620/files/")
DMD5 = {"ToyCar": "4dec75ca8d9f666aa9e4c1894a740501", "ToyConveyor": "03b6aa1bfd09a39d53af0ba39f71fa91"}
XMD5 = {"ToyCar": "a5d4ccd498af70b04dd74b644b820006", "ToyConveyor": "0e808ac0b761e9fb13d5e0c65caff00d"}
EMD5 = {"ToyCar": "bea2bdd612f616be8f2b8eb087b32c7c", "ToyConveyor": "ccfcc9847c7d3404d7ee9d6d9e0b2ba2"}
DC_DEV = {t_: (ZD + f"dev_data_{t_}.zip?download=1", f"dev_data_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_test_{t_}.zip?download=1", f"eval_data_test_{t_}.zip", h) for t_, h in EMD5.items()}
DC_GT = (ZG + "eval_data_list.csv?download=1", "eval_data_list.csv", "f16aaa185df9906e588864aaf7cb58ea")
TYPES = ["ToyCar", "ToyConveyor"]                                     # the two machine types no notebook of this project has read
DEV_IDS = {"ToyCar": ["id_01", "id_02", "id_03", "id_04"], "ToyConveyor": ["id_01", "id_02", "id_03"]}
EVAL_IDS = {"ToyCar": ["id_05", "id_06", "id_07"], "ToyConveyor": ["id_04", "id_05", "id_06"]}
SR, LDN_FLOOR = 16000, 1e-6
NS_RULE = 10 * SR                                                     # every system reads the first 10 s of a clip (ToyCar's clips are about 11 s)
# Part 1 · the frozen recipe: notebook 23's lines, unchanged (notebook 21's settings; the track list fixes the random streams)
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS) + ["DC"]                                            # fixed order for random seeds
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)        # notebook 21's, unchanged
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
B3_SHA_N22B = "ba0f139a46b3a22f4ff78873db92214146cddfaad12db7c026f6dd758df1be6a"
ENCODERS = {"BEATs": {"name": "BEATs iter3+ (AS2M)", "dim": 768, "layers": 12},
            "BEATs-iter3": {"name": "BEATs iter3", "dim": 768, "layers": 12, "file": "BEATs_iter3.pt",
                            "sed_file": "BEATs_iter3_from_PretrainedSED.pt",          # Amendment 1: the third source
                            "sed": "https://github.com/fschmid56/PretrainedSED/releases/download/v0.0.1/BEATs_ssl.pt",
                            "share": "https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5",
                            "azure": ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt?sv=2020-08-04"
                                      "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")},
            "EAT-base": {"name": "EAT-base (epoch 30, pre-trained)", "dim": 768, "layers": 12, "repo": "worstchan/EAT-base_epoch30_pretrain"},
            "EAT-large": {"name": "EAT-large (epoch 20, pre-trained)", "dim": 1024, "layers": 24, "repo": "worstchan/EAT-large_epoch20_pretrain"}}
NEW_ENCODERS = ["BEATs-iter3"]                                       # notebook 23 needs only BEATs iter3 (MIC+ and MIC)
ENC_ORDER = list(ENCODERS)
EAT_NORM, EAT_FRAMES = (-4.268, 4.569), 1024                         # EAT's model card
GB_RATIO = 1.5
FROZEN = [("BEATs", "B", [0, 1, 2]), ("BEATs-iter3", "B3", [0, 1, 2])]   # MIC's final system: encoder, file code, seeds
# Part 2 · the second view: notebook 25's lines, unchanged
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES = 313, (8, 20), 13                # the reference's sizes for 10 s; 7 machines of ToyCar and 6 of ToyConveyor
CAL_MIN = 30                                                          # a spread of form ZP needs at least 30 clips per type
# gate G-DEV: a development AUC below these means a broken pipeline (ToyCar: the DCASE 2020 baseline's; ToyConveyor: a loose line,
# because systems that learn machine identity are weak on that type)
GDEV_BAR = {"ToyCar": 0.7877, "ToyConveyor": 0.55}
GDEV_SYSTEMS = ["ZP", "frozen", "view_paper"]                         # the kept system and its two parts
# the yardsticks of the evaluation set, AUC and pAUC % (the DCASE 2020 results page); the four MIMII types as in notebooks 23 and 25
TEAMS = {"Giri_Amazon_task2_2 (rank 1)": {"fan": (94.54, 84.30), "pump": (93.65, 81.73), "slider": (97.63, 89.73), "valve": (96.13, 90.89),
                                           "ToyCar": (94.34, 89.73), "ToyConveyor": (91.19, 73.34)},
         "Daniluk_SRPOL_task2_4 (rank 3)": {"fan": (99.13, 96.40), "pump": (95.07, 90.23), "slider": (98.18, 91.98), "valve": (90.97, 77.41),
                                             "ToyCar": (93.52, 83.87), "ToyConveyor": (90.51, 77.56)},
         "Primus_CP-JKU_task2_2 (rank 5)": {"fan": (96.84, 95.24), "pump": (97.76, 92.24), "slider": (97.29, 88.74), "valve": (90.15, 86.65),
                                             "ToyCar": (86.37, 83.83), "ToyConveyor": (88.28, 79.15)},
         "Vinayavekhin_IBM_task2_2 (rank 7)": {"fan": (98.84, 94.89), "pump": (94.37, 88.27), "slider": (95.68, 83.09), "valve": (97.82, 94.93),
                                                "ToyCar": (93.16, 87.69), "ToyConveyor": (87.41, 72.03)},
         "Zhou_PSH_task2_3 (rank 12)": {"fan": (99.79, 98.92), "pump": (95.79, 92.60), "slider": (99.84, 99.17), "valve": (91.83, 84.74),
                                         "ToyCar": (95.60, 91.30), "ToyConveyor": (73.61, 64.06)}}
BASELINE_EVAL = {"ToyCar": (80.14, 66.17), "ToyConveyor": (85.36, 66.95)}   # DCASE2020_baseline_task2_1 (official rank 93)
Q1_TEAMS, Q2_BAR = 3, 0.01                                            # "with the teams": above at least 3 of the 5; the fusion's words: 0.01
# question 3: the kept system's evaluation results on the four MIMII types (notebook 25, a second scoring there): AUC, pAUC
N25_EVAL = {'fan': (0.9979608938547487, 0.989365872782515), 'pump': (0.9812022904833624, 0.9201611003549114), 'slider': (0.9992322097378277, 0.9959589986201459), 'valve': (0.9918583333333334, 0.9652192982456139)}
SIX = ["fan", "pump", "slider", "valve", "ToyCar", "ToyConveyor"]
NAME = {"fan": "fan", "pump": "pump", "slider": "slide rail", "valve": "valve", "ToyCar": "ToyCar", "ToyConveyor": "ToyConveyor"}
T3P_BAR = 0.9458                                                      # T3′: the rank-1 team's mean AUC over the six types
# published development numbers, AUC and pAUC % (all six types trained together; yardsticks, not bars)
PUBLISHED_DEV = {"Noisy-ArcMix, TASTgram-MFN (arXiv 2310.06364, Table 1)": {"ToyCar": (96.76, 90.11), "ToyConveyor": (77.90, 67.15)},
                 "DCASE 2020 baseline (the task page)": {"ToyCar": (78.77, 67.58), "ToyConveyor": (72.53, 60.43)}}
# AnoPatch with BEATs, arXiv 2406.11364, Table 1: the mean of AUC and pAUC, %
ANOPATCH = {"development": {"ToyCar": 96.10, "ToyConveyor": 73.20}, "evaluation": {"ToyCar": 96.00, "ToyConveyor": 83.74}}
# the official evaluator, pinned to one commit and checked by its SHA-256 (as in notebook 24)
EVALUATOR = ("https://raw.githubusercontent.com/y-kawagu/dcase2020_task2_evaluator/3e5464fe3091cc647015a84475eb89c3232459b6/evaluator.py",
             "evaluator.py", "1a514d3809b6aabab60db7fa4067d0e59a97c98c728918427a546763974e106a")
EVAL_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 1e-9, 2.0, 20.0
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"])
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items()} for k_ in ("dev", "add", "eval"))
    DC_GT, EVALUATOR = tuple(_dc["gt"]), tuple(_dc["evaluator"])
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    CFG.update(_c.get("cfg", {})); SV.update(_c.get("sv", {}))
    for k_, v_ in _c.get("encoders", {}).items():
        ENCODERS[k_].update(v_)
    CAL_MIN, NS_RULE = _c.get("cal_min", CAL_MIN), _c.get("clip_samples", NS_RULE)
    GDEV_BAR = _c.get("gdev_bar", GDEV_BAR)                           # the synthetic clips have their own bars
FROZEN_CAP = CFG["xc_cap"]                                            # the frozen recipe trains on the first 600 additional clips of a machine
assert [sd for _, _, sd in FROZEN] == [CFG["se_seeds"]] * 2 and ENC_ORDER[:2] == [e for e, _, _ in FROZEN]
print("rules:", RULES, "·", json.dumps(CFG), "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")
if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
    raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")

In [ ]:
# 3 · The official BEATs code (notebook 23's cell, unchanged: the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 4 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 5 · Model pieces (notebook 23's cell, unchanged; notebook 2's code): our forward pass, LoRA, the head and the sub-center ArcFace layer
def encode(x, amp):
    """Waveforms (B, N) in [-1, 1] → mean over time of the last layer (B, 768), float32.
    Same steps as BEATs.extract_features without padding; the filter banks are always computed in float32."""
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # (B, frames, 128)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1)


class LoRALinear(nn.Module):
    """y = W x + (α / r) · B A x. W stays frozen; B starts at zero, so the model starts exactly as the frozen one."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(base.out_features, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class Head(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bn = nn.BatchNorm1d(d_out)

    def forward(self, x):
        return self.bn(self.lin(x))


class SubCenterArcFace(nn.Module):
    """Cosine to the closest of k centres per class; additive angular margin m on the true class; scale s."""
    def __init__(self, d, n_classes, k, s, m):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes * k, d))
        nn.init.xavier_uniform_(self.W)
        self.n, self.k, self.s, self.m = n_classes, k, s, m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def cosine(self, e):
        c = F.linear(F.normalize(e.float(), dim=1), F.normalize(self.W.float(), dim=1))
        return c.view(-1, self.n, self.k).max(dim=2).values

    def loss(self, cos, y):
        sin = torch.sqrt((1.0 - cos * cos).clamp(0.0, 1.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)      # keeps the target logit monotonic near θ = π
        one = F.one_hot(y, self.n).to(cos.dtype)
        return F.cross_entropy(self.s * (one * phi + (1.0 - one) * cos), y)


def equal_length_batches(clips, idx, batch):
    """Consecutive clips of the same length, at most `batch` per group (the same grouping as notebook 1)."""
    buf = []
    for i in idx:
        if buf and len(clips[i]) != len(clips[buf[0]]):
            yield buf; buf = []
        buf.append(i)
        if len(buf) == batch:
            yield buf; buf = []
    if buf:
        yield buf


def as_float(clips, ids):
    return torch.from_numpy(np.stack([clips[i] for i in ids]).astype(np.float32) / 32768.0).to(DEVICE)

In [ ]:
# 6 · LoRA on the query and value projections of all 12 layers, with one switch (PD's frozen teacher runs with it off); checks; a timed step
for p_ in model.parameters():
    p_.requires_grad_(False)                                          # the original weights never change


class LoRASwitch(LoRALinear):
    on = True                                                         # one switch for every block

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale if LoRASwitch.on else y


def tokens(x, amp):
    # waveforms (B, N) in [-1, 1] → last-layer tokens (B, T, 768), float32: notebook 2's encode without the final mean
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float()


rng0 = np.random.default_rng(0)
CROP = int(CFG["crop_s"] * SR)
x4 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (4, CROP)).astype(np.float32)).to(DEVICE)
with torch.no_grad():
    Z4 = tokens(x4, amp=False).mean(1)
    ref = model.extract_features(x4)[0].mean(1)
    gap_e = float((encode(x4, amp=False) - Z4).abs().max())
gap = float((Z4 - ref).abs().max())
print(f"our forward pass against BEATs.extract_features: largest difference {gap:.2e} (notebook 2's encode: {gap_e:.1e})")
assert gap < 1e-3 and gap_e < 1e-5, "stop: our forward pass does not reproduce BEATs"
LORA = []
for layer in model.encoder.layers:
    att = layer.self_attn
    for name in ("q_proj", "v_proj"):
        w = LoRASwitch(getattr(att, name), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"])
        setattr(att, name, w); LORA.append(w)
with torch.no_grad():
    gap = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
print(len(LORA), "LoRA blocks · with LoRA at zero, largest difference from frozen BEATs:", f"{gap:.2e}")
assert gap < 1e-5, "stop: adding LoRA changed the model before training"


def lora_params():
    return [p for w in LORA for p in (w.A, w.B)]


def fresh_adapters(seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen BEATs
    torch.manual_seed(seed)
    for w in LORA:
        w.reset()
    head = Head(768, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


head_t, arc_t = fresh_adapters(12345, 8)
opt_t = torch.optim.AdamW(lora_params() + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
model.train(); head_t.train(); arc_t.train(); times = []
for k in range(3):
    xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
    yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    loss = arc_t.loss(arc_t.cosine(head_t(tokens(xb, amp=AMP).mean(1))), yb)
    opt_t.zero_grad(set_to_none=True); loss.backward()
    if k == 0:
        assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in LORA if w.B.grad is not None) > 0, "stop: no gradient reaches LoRA"
    opt_t.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
model.eval()
with torch.no_grad():
    moved = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = False
    off = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = True
print(f"after 3 steps: with LoRA on the output moved by {moved:.2e}; with the switch off it differs from frozen BEATs by {off:.2e}")
assert moved > 1e-6 and off < 1e-5, "stop: the LoRA switch does not work"
del head_t, arc_t, opt_t
for w in LORA:
    w.reset()
STEP_S = min(times[1:])
print(f"BEATs iter3+ (AS2M): one training step {STEP_S:.3f} s")

In [ ]:
# 7 · BEATs iter3 (notebook 23's cell, unchanged; notebook 22b's loader and file): load, smoke test, LoRA, GB, a timed step; gate G0(h)
import torchaudio.compliance.kaldi as ta_kaldi


def beats_fb_tokens(m, fb, amp):
    # filter banks (B, frames, 128) of any BEATs model → last-layer tokens (B, T·8, 768): the forward pass of cell 9
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = m.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = m.layer_norm(f)
        if m.post_extract_proj is not None:
            f = m.post_extract_proj(f)
        h, _ = m.encoder(m.dropout_input(f), padding_mask=None)
    return h.float()


def kaldi_frames(n):
    # frames of a 25 ms / 10 ms Kaldi filter bank of n samples (edges snipped)
    return 1 + (n - 400) // 160


class LoRAQKV(nn.Module):
    """A fused query-key-value projection with LoRA on the query and value parts only: y = W x + (α / r) · [B_q A_q x, 0, B_v A_v x].
    B starts at zero, so the model starts exactly as the frozen one; LoRASwitch.on switches every block at once."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        d = base.in_features
        assert base.out_features == 3 * d, (base.in_features, base.out_features)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(2, r, d, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(2, d, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        with torch.no_grad():
            for i in range(2):
                nn.init.kaiming_uniform_(self.A[i], a=math.sqrt(5))
            self.B.zero_()

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        y = self.base(x)
        if not LoRASwitch.on:
            return y
        h = self.drop(x)
        dq = F.linear(F.linear(h, self.A[0]), self.B[0]) * self.scale
        dv = F.linear(F.linear(h, self.A[1]), self.B[1]) * self.scale
        return y + torch.cat([dq, torch.zeros_like(dq), dv], -1)


class Enc:
    # one encoder: waveforms → last-layer patch tokens (no CLS), its LoRA blocks, its patch layout, and pooling over the time
    # patches that start inside the clip
    def __init__(self, key, net, lora, info=None):
        self.key, self.name, self.dim, self.net, self.lora = key, ENCODERS[key]["name"], ENCODERS[key]["dim"], net, list(lora)
        self.layout, self.ok_lora, self.step_s, self.info = "frequency fastest", bool(lora), None, dict(info or {})

    def tokens(self, x, amp):
        with torch.autocast(device_type=DEVICE, enabled=False):
            fb = self.wave2fb(x.float())                              # filter banks always in float32
        return self.fb2tok(fb, amp)

    def grid(self, h, n_samples):
        # (B, L, D) tokens → (B, T, 8, D): the time patches that start inside a clip of n_samples, in this encoder's layout
        B, Lt, D = h.shape; Tp = Lt // 8
        g = h.view(B, Tp, 8, D) if self.layout == "frequency fastest" else h.view(B, 8, Tp, D).transpose(1, 2)
        return g[:, :max(1, min(Tp, -(-kaldi_frames(n_samples) // 16)))]

    def pool(self, h, n_samples):
        return self.grid(h, n_samples).mean((1, 2))

    def train(self):
        self.net.train()

    def eval(self):
        self.net.eval()

    def reset_lora(self):
        for w in self.lora:
            w.reset()


class BeatsEnc(Enc):
    def wave2fb(self, x):
        return self.net.preprocess(x)

    def fb2tok(self, fb, amp):
        return beats_fb_tokens(self.net, fb, amp)


class EatEnc(Enc):
    varlen = False

    def wave2fb(self, x):
        # EAT's model card: waveform minus its mean; Kaldi filter banks (128 mel, Hanning, 10 ms shift, no dither, HTK-compatible);
        # zero-padded to 1,024 frames (training crops: to a multiple of 16 frames when the model accepts it); then normalised
        out = []
        for w in x:
            w = w - w.mean()
            out.append(ta_kaldi.fbank(w.unsqueeze(0), htk_compat=True, sample_frequency=SR, use_energy=False, window_type="hanning",
                                      num_mel_bins=128, dither=0.0, frame_shift=10))
        fb = torch.stack(out); n = fb.shape[1]
        tgt = EAT_FRAMES if (n > EAT_FRAMES - 128 or not self.varlen) else 16 * math.ceil(n / 16)
        fb = F.pad(fb, (0, 0, 0, tgt - n)) if tgt >= n else fb[:, :tgt]
        return (fb - EAT_NORM[0]) / (2.0 * EAT_NORM[1])

    def fb2tok(self, fb, amp):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
            out = self.net.extract_features(fb.unsqueeze(1))
        if isinstance(out, dict):
            out = out["x"] if "x" in out else next(iter(out.values()))
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out[:, 1:].float()                                     # without the CLS token


class StandInAttn(nn.Module):
    # TEST only: attention with one fused qkv projection, like EAT's
    def __init__(self, dim, heads=2):
        super().__init__()
        self.h, self.qkv, self.proj = heads, nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))


class StandInBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.n1, self.attn, self.n2 = nn.LayerNorm(dim), StandInAttn(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))


class EatStandIn(nn.Module):
    # TEST only: a small ViT with EAT's interface: extract_features([B, 1, T, 128]) → CLS + (T/16 × 8) patch tokens
    def __init__(self, dim, layers):
        super().__init__()
        self.patch, self.cls = nn.Conv2d(1, dim, 16, 16), nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, 64 * 8, dim) * 0.02)
        self.blocks, self.norm = nn.ModuleList(StandInBlock(dim) for _ in range(layers)), nn.LayerNorm(dim)

    def extract_features(self, mel):
        x = self.patch(mel).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x + self.pos[:, :x.shape[1]]], 1)
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


def onedrive_direct(share_url):
    # OneDrive share link → direct download (the documented 'shares' form of the OneDrive API), as notebook 1
    b64 = base64.b64encode(share_url.encode()).decode().rstrip("=").replace("/", "_").replace("+", "-")
    return f"https://api.onedrive.com/v1.0/shares/u!{b64}/root/content"


def attach_beats_lora(m):
    lora = []
    for layer in m.encoder.layers:
        att = layer.self_attn
        for nm in ("q_proj", "v_proj"):
            w = LoRASwitch(getattr(att, nm), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(att, nm, w); lora.append(w)
    return lora


WN = ((".parametrizations.weight.original0", ".weight_g"), (".parametrizations.weight.original1", ".weight_v"))


def tensors_of(ck):
    # the weights of a loaded checkpoint (a bare state dict, or under "model" or "state_dict") and its configuration, if it has one
    if isinstance(ck, dict):
        for k in ("model", "state_dict"):
            if isinstance(ck.get(k), dict):
                return ck[k], ck.get("cfg")
        if ck and all(torch.is_tensor(v) for v in ck.values()):
            return ck, None
    raise RuntimeError(f"no weights in the file (it holds a {type(ck).__name__}" + (f" with keys {list(ck)[:5]})" if isinstance(ck, dict) else ")"))


def wn_names(sd, own_keys):
    # torch names weight norm's two tensors in two ways (weight_g / weight_v, and parametrizations); the file's keys take the model's
    old = any(k.endswith(".weight_g") for k in own_keys)
    out = {}
    for k, v in sd.items():
        for a, b in WN:
            k = k.replace(a, b) if old else k.replace(b, a)
        out[k] = v
    return out


def match_weights(net, sd):
    # Amendment 1: the file's keys on the model's own, as they are or with one leading prefix removed or added (whichever matches
    # the most keys). Every trainable parameter must come from the file; fixed tensors (buffers, and parameters without gradient)
    # may come from the model's own construction; keys of the file that the model does not have are not used. Both are listed.
    own = net.state_dict(); keys = set(own)
    trainable = {n for n, q in net.named_parameters() if q.requires_grad}
    sd = wn_names(sd, keys)
    cands = {("as is", "")}
    for k in sd:
        for o in keys:
            if k.endswith("." + o):
                cands.add(("removed", k[:len(k) - len(o)]))
            elif o.endswith("." + k):
                cands.add(("added", o[:len(o) - len(k)]))

    def mapping(how, p):
        return {x[len(p):]: x for x in sd if x.startswith(p)} if how == "removed" else {p + x: x for x in sd}

    how, p = max(sorted(cands), key=lambda c: (len(set(mapping(*c)) & keys), c[0] == "as is"))
    m = {k: x for k, x in mapping(how, p).items() if k in keys}
    other = [k for k in m if tuple(sd[m[k]].shape) != tuple(own[k].shape)]
    missing = sorted(keys - set(m)); lost = [k for k in missing if k in trainable]
    unused = sorted(set(sd) - set(m.values()))
    if other or lost:
        raise RuntimeError(f"the file's weights do not fit the model: {len(lost)} trainable parameters are not in the file (e.g. {lost[:3]}), "
                           f"{len(other)} have another shape (e.g. {other[:3]})")
    rep = {"keys_loaded": len(m), "key_prefix": f"'{p}' {how}" if p else "none", "fixed_tensors_from_construction": missing,
           "file_keys_not_used": len(unused), "file_keys_not_used_e_g": unused[:5]}
    return {k: sd[x] for k, x in m.items()}, rep


def load_matched(net, sd):
    state, rep = match_weights(net, sd)
    res = net.load_state_dict(state, strict=False)
    assert not res.unexpected_keys and sorted(res.missing_keys) == rep["fixed_tensors_from_construction"], res
    return rep


def beats_iter3_file():
    # a BEATs iter3 file already in Drive is used as it is; otherwise the authors' link, the Azure copy, then PretrainedSED's copy
    spec = ENCODERS["BEATs-iter3"]; d = WORK if TEST else CKPT_DIR
    own, sed = d / spec["file"], d / spec["sed_file"]
    for f in (own, sed):
        if f.exists() and f.stat().st_size < 10_000_000:
            f.unlink()                                                # an error page saved by a failed attempt, not the model
    if not (own.exists() or sed.exists()):
        for label, url, dest in (("the authors' link (OneDrive)", spec["share"] and onedrive_direct(spec["share"]), own),
                                 ("Microsoft's Azure copy", spec["azure"], own), ("PretrainedSED's release v0.0.1", spec["sed"], sed)):
            if not url:
                continue
            try:
                download(url, dest)
            except Exception as e:
                print(f"BEATs iter3 · {label}: failed ({e.__class__.__name__})")
            if dest.exists() and dest.stat().st_size < 10_000_000:
                dest.unlink()
            if dest.exists():
                print(f"BEATs iter3 · {label}: downloaded", flush=True)
                break
    for f, label in ((own, "the authors' file"), (sed, "PretrainedSED's copy (release v0.0.1); provenance caveat in Amendment 1")):
        if f.exists():
            return f, label
    raise RuntimeError("no download of BEATs iter3 worked (the authors' link, the Azure copy, PretrainedSED's copy)")


def load_beats_iter3():
    f, label = beats_iter3_file()
    ck = torch.load(f, map_location="cpu", weights_only=True)          # tensors and plain values only: no code in the file runs
    sd, cfg_f = tensors_of(ck)
    cfg_ = cfg_f if cfg_f is not None else {**chk["cfg"], "finetuned_model": False}   # weights only: our iter3+ architecture, no classifier
    m = BEATs(BEATsConfig(cfg_))
    rep = load_matched(m, sd)
    msd, ref = m.state_dict(), chk["model"]
    com = [k for k in msd if k in ref and tuple(ref[k].shape) == tuple(msd[k].shape)]
    dist = math.sqrt(sum(float((msd[k].double() - ref[k].double()).pow(2).sum()) for k in com)
                     / max(sum(float(ref[k].double().pow(2).sum()) for k in com), 1e-30))
    m.eval().to(DEVICE)
    for q in m.parameters():
        q.requires_grad_(False)
    info = {"checkpoint_sha256": sha256(f), "file": f.name, "source": label,
            "configuration": "the file's own" if cfg_f is not None else "our iter3+ checkpoint's, classifier off (the file holds weights only)",
            "fine_tuned_on_audioset": bool(getattr(BEATsConfig(cfg_), "finetuned_model", False)),
            "relative_distance_to_our_iter3plus": round(dist, 4), "tensors_compared": len(com), **rep}
    print(f"BEATs iter3 · {label} · SHA-256 {info['checkpoint_sha256'][:16]}… · relative distance from our iter3+ weights {dist:.4f}")
    return BeatsEnc("BEATs-iter3", m, attach_beats_lora(m), info)


def eat_snapshot(repo):
    # the repository's files at one commit: configuration, code and weights (TEST: a local folder)
    if os.path.isdir(repo):
        return repo, "a local folder (TEST)"
    from huggingface_hub import HfApi, snapshot_download
    names = HfApi().list_repo_files(repo)
    pats = ["*.json", "*.py", "*.txt"] + (["*.safetensors"] if any(n.endswith(".safetensors") for n in names) else ["*.bin"])
    path = snapshot_download(repo, allow_patterns=pats)
    return path, Path(path).name                                      # the snapshot folder is named by its commit


def read_weights(path):
    # every weights file of the snapshot: safetensors, else pytorch_model*.bin read with weights_only=True
    files, sd = sorted(Path(path).glob("*.safetensors")), {}
    if files:
        from safetensors.torch import load_file
        for f in files:
            sd.update(load_file(str(f)))
    else:
        files = sorted(Path(path).glob("pytorch_model*.bin"))
        for f in files:
            sd.update(torch.load(f, map_location="cpu", weights_only=True))
    if not sd:
        raise RuntimeError("no weights file (safetensors or pytorch_model.bin) in the repository")
    return sd, [f.name for f in files]


def load_remote_model(path):
    # from_pretrained first; if it fails (transformers 5 with remote code that does not call post_init), the model is built from
    # the repository's own configuration and code, and its weights file is loaded into it (Amendment 1). A package that its
    # code needs is installed, then the load is tried again.
    from transformers import AutoConfig, AutoModel
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    for _ in range(4):
        try:
            try:
                return AutoModel.from_pretrained(path, trust_remote_code=True), {"loaded_by": "from_pretrained"}
            except ImportError:
                raise
            except Exception as e:
                first = repr(e)[:300]
            print(f"  from_pretrained failed ({first[:140]}); building the model from its own configuration and code", flush=True)
            conf = AutoConfig.from_pretrained(path, trust_remote_code=True)
            net = get_class_from_dynamic_module(conf.auto_map["AutoModel"], path)(conf)
            sd, files = read_weights(path)
            rep = load_matched(net, sd)
            return net, {"loaded_by": "built from its own configuration and code; weights loaded directly", "from_pretrained_error": first,
                         "weights_files": files, **rep}
        except ImportError as e:
            if isinstance(e, ModuleNotFoundError) and e.name:
                need = [e.name.split(".")[0]]
            else:
                need = " ".join(re.findall(r"pip install ([^`\n]+)", str(e))[:1]).split()
            if not need:
                raise
            print(f"  its code needs {need}; installing", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
            importlib.invalidate_caches()
    raise RuntimeError("did not load")


def load_eat(key):
    spec = ENCODERS[key]
    if TEST and not os.path.isdir(str(spec.get("repo"))):
        torch.manual_seed(ENC_ORDER.index(key)); net, info = EatStandIn(spec["dim"], spec["layers"]), {"commit": "TEST stand-in"}
    else:
        try:
            import transformers
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
            import transformers
        path, commit = eat_snapshot(spec["repo"])
        net, info = load_remote_model(path)
        info.update(commit=commit, transformers=transformers.__version__)
    net = net.float().to(DEVICE).eval()
    for q in net.parameters():
        q.requires_grad_(False)
    info = {"repo": spec.get("repo"), **info, "parameters_millions": round(sum(q.numel() for q in net.parameters()) / 1e6, 1)}
    fused = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] == "qkv"]
    split = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in ("q_proj", "v_proj", "query", "value")]
    lora = []
    if len(fused) == spec["layers"]:
        for n, m in fused:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRAQKV(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(fused)} fused query-key-value projections (query and value parts)"
    elif len(split) == 2 * spec["layers"]:
        for n, m in split:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRASwitch(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(split)} separate query and value projections"
    else:
        info["lora_problem"] = f"found {len(fused)} fused qkv and {len(split)} separate query/value projections; {spec['layers']} layers expected"
    return EatEnc(key, net, lora, info)


def lora_params_of(enc):
    return [p for w in enc.lora for p in (w.A, w.B)]


def smoke(enc, timed=True):
    # tokens: shape, finiteness; EAT: other lengths; LoRA: unchanged at zero, a gradient reaches it, the switch works; a timed step
    rep = {}
    x10 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (2, 10 * SR)).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        h = enc.tokens(x10, amp=False)
    tp = EAT_FRAMES // 16 if isinstance(enc, EatEnc) else kaldi_frames(10 * SR) // 16
    rep["tokens_of_10_s"] = list(h.shape)
    if not (h.shape[1] == tp * 8 and h.shape[2] == enc.dim and bool(torch.isfinite(h).all())):
        raise RuntimeError(f"{enc.key}: tokens {tuple(h.shape)}, expected (2, {tp * 8}, {enc.dim}) and finite")
    if isinstance(enc, EatEnc):
        enc.varlen = True
        try:
            with torch.no_grad():
                hc = enc.tokens(x10[:, :CROP], amp=False)
            enc.varlen = bool(hc.shape[1] == 16 * math.ceil(kaldi_frames(CROP) / 16) // 16 * 8 and torch.isfinite(hc).all())
        except Exception as e:
            enc.varlen = False; rep["other_length_error"] = repr(e)[:200]
        rep["accepts_other_lengths"] = enc.varlen
    if not enc.lora:
        enc.ok_lora = False; rep["lora"] = enc.info.get("lora_problem", "no LoRA"); return rep
    try:
        xs = x10[:, :CROP]
        with torch.no_grad():
            LoRASwitch.on = False; z0 = enc.pool(enc.tokens(xs, amp=False), CROP)
            LoRASwitch.on = True; z1 = enc.pool(enc.tokens(xs, amp=False), CROP)
        scale = float(z0.abs().mean()) + 1e-12
        assert float((z1 - z0).abs().max()) / scale < 1e-4, "LoRA at zero changed the output"
        head_t = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
        arc_t = SubCenterArcFace(CFG["emb_dim"], 8, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
        opt_t = torch.optim.AdamW(lora_params_of(enc) + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
        enc.train(); head_t.train(); arc_t.train(); times = []
        for k in range(3):
            xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
            yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            loss = arc_t.loss(arc_t.cosine(head_t(enc.pool(enc.tokens(xb, amp=AMP), CROP))), yb)
            opt_t.zero_grad(set_to_none=True); loss.backward()
            if k == 0:
                assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in enc.lora if w.B.grad is not None) > 0, "no gradient reaches LoRA"
            opt_t.step()
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        enc.eval()
        with torch.no_grad():
            moved = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = False
            off = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = True
        assert moved > 1e-6 and off < 1e-4, f"the LoRA switch does not work (moved {moved:.1e}, off {off:.1e})"
        enc.step_s = min(times[1:]) if timed else None
        rep["lora"] = enc.info.get("lora_on", "LoRA"); rep["step_seconds"] = round(enc.step_s, 4) if enc.step_s else None
        del head_t, arc_t, opt_t
    except Exception as e:
        enc.ok_lora = False; rep["lora"] = f"failed: {repr(e)[:300]}"
    finally:
        LoRASwitch.on = True; enc.reset_lora(); enc.eval()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rep


def gate_gb(enc):
    # GB: extra energy in the top (bottom) 16 mel bins over the middle third, against the same input without it; the change of
    # each token, averaged per frequency row under each candidate layout
    with torch.no_grad():
        base = enc.wave2fb(torch.from_numpy(np.random.default_rng(5).normal(0, 0.003, (1, 10 * SR)).astype(np.float32)).to(DEVICE))
        a, b = base.shape[1] // 3, 2 * base.shape[1] // 3
        top, bot = base.clone(), base.clone()
        top[:, a:b, 112:128] += 2.0; bot[:, a:b, 0:16] += 2.0
        LoRASwitch.on = False
        try:
            h0, ht, hb = (enc.fb2tok(z, amp=False) for z in (base, top, bot))
        finally:
            LoRASwitch.on = True
    Lt = h0.shape[1]; Tp = Lt // 8
    dt = (ht - h0).norm(dim=-1)[0].cpu().numpy(); db = (hb - h0).norm(dim=-1)[0].cpu().numpy()
    out = {}
    for lay, row in (("frequency fastest", np.arange(Lt) % 8), ("time fastest", np.arange(Lt) // Tp)):
        pt = np.array([dt[row == f].mean() for f in range(8)]); pb = np.array([db[row == f].mean() for f in range(8)])
        it, ib = int(pt.argmax()), int(pb.argmax())
        rt = float(pt[it] / max(np.delete(pt, it).mean(), 1e-12)); rb = float(pb[ib] / max(np.delete(pb, ib).mean(), 1e-12))
        out[lay] = {"top_row": it, "bottom_row": ib, "ratio_top": round(rt, 2), "ratio_bottom": round(rb, 2),
                    "confirmed": bool({it, ib} == {0, 7} and min(rt, rb) >= GB_RATIO)}   # the two ends of the spectrum, one each
    if out["time fastest"]["confirmed"] and not out["frequency fastest"]["confirmed"]:
        enc.layout = "time fastest"
    out["used"] = enc.layout
    return out


ENC, FAILED, GBR = {}, {}, {}
ENC["BEATs"] = BeatsEnc("BEATs", model, LORA, {"checkpoint_sha256": CKPT_SHA})        # cell 9's model and LoRA blocks
ENC["BEATs"].step_s = STEP_S
with torch.no_grad():
    _gap = float((ENC["BEATs"].tokens(x4, amp=False) - tokens(x4, amp=False)).abs().max())
    _pool = float((ENC["BEATs"].pool(tokens(x4, amp=False), x4.shape[1]) - tokens(x4, amp=False).mean(1)).abs().max())
print(f"BEATs through the encoder wrapper against cell 9's forward pass: {_gap:.1e} (pooling {_pool:.1e})")
assert _gap < 1e-6 and _pool < 1e-5, "stop: the encoder wrapper does not reproduce cell 9"
for key in NEW_ENCODERS:
    t0 = time.time()
    try:
        enc = load_beats_iter3() if key == "BEATs-iter3" else load_eat(key)
        enc.info.update(smoke(enc))
        ENC[key] = enc
        print(f"{key}: ready · {json.dumps(enc.info)} · {time.time() - t0:.0f} s", flush=True)
    except Exception as e:
        FAILED[key] = repr(e)[:500]
        print(f"{key}: DROPPED, our loader failed ({FAILED[key]})", flush=True)
    gc.collect()
for key, enc in ENC.items():
    GBR[key] = gate_gb(enc)
    print(f"GB · {key}: {json.dumps(GBR[key])}")


# G0(h): the BEATs iter3 file is the one notebook 22b used
G0H = {"sha256": ENC["BEATs-iter3"].info.get("checkpoint_sha256") if "BEATs-iter3" in ENC else None, "expected": B3_SHA_N22B}
print("G0(h) · the BEATs iter3 file is notebook 22b's:", G0H["sha256"] == B3_SHA_N22B)
if not TEST and G0H["sha256"] != B3_SHA_N22B:
    raise SystemExit("stop: G0(h) failed, the BEATs iter3 file is not the one notebook 22b used (or it did not load)")

In [ ]:
# 8 · ECHO, frozen (notebook 23's cell, unchanged; notebook 20's cell: notebook 15's loader, as on the model card); gate G0(g), its weights
INFO = {}
def load_echo():
    # as on the ECHO model card: its own code and model.safetensors from the same Hugging Face snapshot
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file
    snap = Path(snapshot_download(ECHO_HF))
    INFO["ECHO"] = {"repo": ECHO_HF, "revision": snap.name, "weights_sha256": sha256(snap / "model.safetensors"),
                    "code_sha256": sha256(snap / "audioMAE_band_upgrade.py")}
    saved = {k: sys.modules.pop(k) for k in ("modules", "utils") if k in sys.modules}
    sys.path.insert(0, str(snap))
    try:
        mod = importlib.import_module("audioMAE_band_upgrade")
    finally:
        sys.path.remove(str(snap))
        for k in ("modules", "utils"):
            sys.modules.pop(k, None)
        sys.modules.update(saved)
    m = mod.AudioMAEWithBand(spec_len=2000, band_width=32, shift_size=16, in_chans=1, embed_dim=384, encoder_depth=12,
                             num_heads=6, mlp_ratio=4.0, freq_pos_emb_dim=384)
    missing, unexpected = m.load_state_dict(load_file(str(snap / "model.safetensors")), strict=False)
    assert not missing, f"stop: ECHO weights missing for {list(missing)[:5]}"
    INFO["ECHO"].update(loader="model card: its own code + model.safetensors", unexpected_keys=len(unexpected))
    return m.to(DEVICE).eval()




@torch.no_grad()
def emb_echo(m, waves):
    out = []
    for x in waves:
        u, _ = m.extract_features_from_audio(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[None].to(DEVICE), sample_rate=SR)
        out.append(u.float().cpu().numpy())
    return np.stack(out)


class EchoStandIn(torch.nn.Module):
    # TEST ONLY: a fixed random projection of log band energies, shaped like ECHO's output. Never used on Colab.
    def __init__(self, seed=2, bands=7, dim=16):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.bands, self.P = bands, torch.randn(bands, 8, dim, generator=g)

    def extract_features_from_audio(self, x, sample_rate):
        S = torch.stft(x[0].float(), n_fft=400, hop_length=160, window=torch.hann_window(400, device=x.device), return_complex=True).abs().log1p().T
        e = torch.stack([b.mean(-1) for b in S.chunk(self.bands, dim=-1)], -1)
        q = torch.stack([e[i * e.shape[0] // 8:(i + 1) * e.shape[0] // 8].mean(0) for i in range(8)], 0)
        return torch.einsum("qk,kqd->kd", q, self.P.to(x.device)).flatten(), None


if TEST:
    ECHO = EchoStandIn().to(DEVICE).eval(); INFO["ECHO"] = {"repo": "stand-in (TEST)", "weights_sha256": "", "loader": "stand-in"}
else:
    ECHO = load_echo()
    if INFO["ECHO"]["weights_sha256"] != ECHO_SHA_N15:
        print("NOTE: the ECHO weights differ from notebook 15's (gate G0(g))")

G0G = {"weights_sha256": INFO["ECHO"].get("weights_sha256"), "expected": ECHO_SHA_N15, "revision": INFO["ECHO"].get("revision")}
print("ECHO:", json.dumps(INFO["ECHO"]))
if not TEST and G0G["weights_sha256"] != ECHO_SHA_N15:
    raise SystemExit("stop: G0(g) failed, the ECHO weights are not notebook 15's")


@torch.no_grad()
def echo_stream(mm, rows=None, chunk=64):
    # ECHO on whole clips of a stream, a few at a time → (n, 7 × 384) float32
    rows = np.arange(mm.shape[0]) if rows is None else rows
    return np.concatenate([emb_echo(ECHO, [f32(mm[r]) for r in rows[i:i + chunk]]) for i in range(0, len(rows), chunk)])

# gate G0(c) of this notebook's rules: the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's (SHA-256).
# The two checkpoints and ECHO were already checked by the cells above; the code files are checked here
G0C = {"code_files": {f: bool(code_sha[f] == h) for f, h in CODE_SHA256.items()}, "beats_iter3plus": bool(CKPT_SHA == CKPT_SHA_N1),
       "beats_iter3": bool(G0H["sha256"] == B3_SHA_N22B), "echo_weights": bool(G0G["weights_sha256"] == ECHO_SHA_N15)}
G0C["ok"] = bool(all(G0C["code_files"].values()) and G0C["beats_iter3plus"] and G0C["beats_iter3"] and G0C["echo_weights"])
print("G0(c) · the BEATs code files, both BEATs checkpoints and ECHO's weights are notebook 23's:", G0C["ok"], G0C)
if not TEST and not G0C["ok"]:
    raise SystemExit("stop: G0(c) failed, a code file or a model file is not the one notebook 23 used")

In [ ]:
# 9 · DCASE 2020's own files of the two types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store on the
# local disk; gates G0(a) and G0(b). The readers and the loader are notebook 23's (through notebook 25); the loader has these types'
# machine IDs, the cut to 10 s, and a check of every file.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores27"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2020 clip of type t in one int16 store, cut to its first NS_RULE samples (10 s, as the view's reference code cuts
    # it): development train and test, the additional training clips (all of them) and the evaluation test clips
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {"dev": zipfile.ZipFile(get_file(DC_DEV[t])), "evaltrain": zipfile.ZipFile(get_file(XZIPS[t])), "evaltest": zipfile.ZipFile(get_file(DC_EVAL[t]))}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            part = "train" if folder == "train" else "test"
            label = (1 if base.startswith("anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(id_\d+)", base).group(1), label, base))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The machine IDs of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and not r[6].startswith("normal") for r in rows), "the additional training clips are normal"
    ns, lengths = NS_RULE, Counter()
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= ns):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {ns / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        mm[i] = to_int16(x[:ns, 0])
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in (DC_DEV[t], XZIPS[t], DC_EVAL[t]):
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}}
for t in TYPES:
    dcase_load(t)
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["lengths"][t] = DMETA[t]["lengths"]
    print(f"DCASE 2020 · {t}: {len(DMETA[t]['name'])} clips {DCASE['counts'][t]} · samples per clip before the cut (set|part: samples: clips): "
          f"{DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print("G0(a) · the six zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def g0b(t):
    # the files are what the rules assume: the machine IDs, development machines with training clips and test clips of both kinds,
    # normal additional-training clips, evaluation test names without a label (16 kHz, one channel and 10 s were checked per file)
    m = DMETA[t]; nm, st, pt, lb, mid = (np.array(m[k]) for k in ("name", "set", "part", "label", "mid"))
    c = DCASE["counts"][t]; ev_test = nm[(st == "eval") & (pt == "test")]
    out = {"ids": bool(sorted(c["dev|train"]) == sorted(c["dev|test"]) == DEV_IDS[t] and sorted(c["eval|train"]) == sorted(c["eval|test"]) == EVAL_IDS[t]),
           "development_machines_have_both_kinds": bool(all(int(((st == "dev") & (pt == "test") & (mid == q) & (lb == k_)).sum()) > 0
                                                            for q in DEV_IDS[t] for k_ in (0, 1))),
           "training_clips_are_normal": bool(all(n_.startswith("normal_") for n_ in nm[pt == "train"])),
           "development_test_names_carry_the_label": bool(all(n_.startswith(("normal_", "anomaly_")) for n_ in nm[(st == "dev") & (pt == "test")])),
           "evaluation_test_names_carry_no_label": bool(len(ev_test) > 0 and all(n_.startswith("id_") and "normal" not in n_ and "anomaly" not in n_ for n_ in ev_test)),
           "no_evaluation_label_in_the_store": bool((lb[(st == "eval") & (pt == "test")] == -1).all()),
           "names_are_unique_in_each_folder": bool(all(len(set(nm[(st == s_) & (pt == p_)].tolist())) == int(((st == s_) & (pt == p_)).sum())
                                                       for s_ in ("dev", "eval") for p_ in ("train", "test"))),
           "clips": int(len(nm)), "skipped_archive_entries": int(len(m["skipped_archive_entries"])), "shortest_clip_samples": int(min(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "longest_clip_samples": int(max(int(k_) for d_ in m["lengths"].values() for k_ in d_))}
    out["ok"] = bool(all(v for k, v in out.items() if isinstance(v, bool)) and out["shortest_clip_samples"] >= NS)
    return out


G0["b"] = {t: g0b(t) for t in TYPES}
print("G0(b) · the machine IDs, the clip kinds and the names are the rules'; every clip is 16 kHz, one channel, at least 10 s:",
      all(v["ok"] for v in G0["b"].values()), G0["b"])
if not all(v["ok"] for v in G0["b"].values()):
    raise SystemExit("stop: G0(b) failed, the files are not what the rules assume")
PREP_S = round(time.time() - t_data, 1)
print(f"data ready · {NS} samples per clip are kept ({NS / SR:g} s) · {PREP_S / 60:.1f} min")

In [ ]:
# 10 · Part 1 · The frozen recipe on these types: notebook 23's Part 2 with the machine types changed and nothing else. DCASE's own
# lists (13 classes), ECHO on every clip, the six trainings (gate G1). Then the models leave the GPU.
# The training functions are notebook 23's whole block, so they still hold its branches for the MIMII tracks and for the band views;
# with the DCASE lists (track "DC") and views=False those branches are never entered.
t_p1 = time.time()
ENCT = {"BEATs": "B", "BEATs-iter3": "B3", "EAT-base": "EB", "EAT-large": "EL"}
TRK = {"ARR": "ARR", "MIC+": "MICp", "MIC": "MIC", "DC": "DC"}


def n21_tag(track, flags, seed, fold):
    return f"{TRK[track]}_{'-'.join(f for f in ('XC', 'PL', 'TS', 'PD') if f in flags) or 'R0'}_s{seed}_f{fold}"


def n22_tag(ek, track, inp, flags, seed, fold):
    return f"{ENCT[ek]}_{TRK[track]}{'n' if inp == 'N2N-MWF' else ''}_{'-'.join(flags) or 'R0'}_s{seed}_f{fold}"


def stream_name(inp, fold):
    # the stream a model of fold k reads: denoised inputs use denoiser k, so no learned part has heard a new machine
    return {"MWF": "MWF_m6", "CH0": "CH0_m6", "N2N-CH0": f"N2N{fold}_m6", "N2N-MWF": f"N2NMWF{fold}_m6"}[inp]


def training_set(track, flags, fold):
    # items (type index, source 0 = MIMII / 1 = XC, row, class): one class per machine of the fold, and per extra machine with XC
    if track == "DC":
        return dcase_training_set()                                   # notebook 23: DCASE 2020's own lists (cell 13b)
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        for p in FOLDS[fold]:
            if p in SPLIT[t]:
                c = len(classes); classes.append(f"{t}|{p}"); items.extend((ti, 0, int(r), c) for r in TRAIN_ROWS[(t, p)])
    if "XC" in flags:
        for ti, t in enumerate(TYPES):
            xm = np.array(XMETA[t]["mid"])
            for q in sorted(set(xm.tolist())):
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020"); items.extend((ti, 1, int(r), c) for r in np.where(xm == q)[0])
    return np.array(items, dtype=np.int64), classes


def win(mm, row, s, n):
    return f32(mm[row, s:s + n])


def make_batch(track, inp, fold, items, ids, rng, n, want_teacher):
    # the student's crops; PD's teacher crops: the 6 dB twin of the same window through MWF (ARR only)
    B = len(ids); x = np.zeros((B, n), np.float32); tch = np.zeros((B, n), np.float32) if want_teacher else None
    has = np.zeros(B, bool)
    for b, it in enumerate(ids):
        ti, source, r, _ = items[it]; t = TYPES[ti]
        if source >= 2:                                               # DCASE 2020 (cell 13b): the clip itself is microphone 0
            mm = DST[t]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        if source == 1:
            mm = XS[t]["CH0" if track == "MIC" else f"N2N{fold}"]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        mm = ST[t][stream_name(inp, fold)]; s = int(rng.integers(0, mm.shape[1] - n + 1))
        x[b] = win(mm, r, s, n)
        if want_teacher:
            assert track == "ARR" and IS_TRAIN[t][r]
            tch[b] = win(ST[t]["MWF_6"], r, s, n); has[b] = True
    T = torch.from_numpy(tch).to(DEVICE) if want_teacher else None
    return torch.from_numpy(x).to(DEVICE), T, torch.from_numpy(has).to(DEVICE)


def fresh(enc, seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen encoder
    torch.manual_seed(seed); enc.reset_lora()
    head = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


def adapter_state(enc, head, arc):
    return {"lora": [{"A": w.A.detach().cpu(), "B": w.B.detach().cpu()} for w in enc.lora],
            "head": {k: v.cpu() for k, v in head.state_dict().items()}, "arc": {k: v.cpu() for k, v in arc.state_dict().items()}}


def load_lora(enc, lst):
    assert len(lst) == len(enc.lora), (len(lst), len(enc.lora))
    with torch.no_grad():
        for w, s in zip(enc.lora, lst):
            w.A.copy_(s["A"].to(DEVICE)); w.B.copy_(s["B"].to(DEVICE))


def teacher_tokens(enc, T):
    # the frozen encoder (LoRA off, evaluation mode, no gradient) on the teacher crops
    with torch.no_grad():
        LoRASwitch.on = False; enc.eval()
        try:
            return enc.tokens(T, amp=AMP)
        finally:
            LoRASwitch.on = True; enc.train()


def step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n):
    tok = enc.tokens(X, amp=AMP)
    cos = arc.cosine(head(enc.pool(tok, n)))
    ce = lam * arc.loss(cos, y) + (1.0 - lam) * arc.loss(cos, y[perm]) if mix else arc.loss(cos, y)
    pd = None
    if T is not None and bool(has.any()):
        tt = teacher_tokens(enc, T[has])
        pd = (1.0 - F.cosine_similarity(tok[has], tt, dim=-1)).mean()
    return ce, pd


def train(ek, track, inp, flags, seed, fold):
    # notebook 21's training for any encoder: LoRA + head + sub-centre ArcFace, mixup, warm-up and cosine decay, PD's λ rule
    enc = ENC[ek]; tag = n22_tag(ek, track, inp, flags, seed, fold)
    final, resume = OUT / f"ft_{tag}.pt", OUT / f"ft_{tag}_resume.pt"
    items, classes = training_set(track, flags, fold)
    head, arc = fresh(enc, seed, len(classes))
    if final.exists():
        st = torch.load(final, map_location="cpu", weights_only=False)
        assert st["classes"] == classes, "the saved model used other classes"
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        return head, st["log"]
    pd_on = "PD" in flags
    params = lora_params_of(enc) + list(head.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW([{"params": lora_params_of(enc), "lr": CFG["lr_lora"], "weight_decay": 0.0},
                             {"params": list(head.parameters()) + list(arc.parameters()), "lr": CFG["lr_head"], "weight_decay": CFG["wd_head"]}])
    spe = len(items) // CFG["batch"]
    total, warm = spe * CFG["epochs"], spe * CFG["warmup_epochs"]
    lr_f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1.0 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, [lr_f, lr_f])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"tag": tag, "encoder": ek, "track": track, "input": inp, "flags": list(flags), "seed": seed, "fold": fold, "cfg": CFG,
           "classes": classes, "n_train": int(len(items)), "n_xc": int(np.isin(items[:, 1], (1, 3)).sum()), "lambda_pd": None,
           "epochs": [], "resumed_at_epoch": []}
    start, n = 0, CROP
    if resume.exists():
        st = torch.load(resume, map_location="cpu", weights_only=False)
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        log, start = st["log"], st["epoch"] + 1
        log["resumed_at_epoch"].append(start); print(f"resuming {tag} at epoch {start}")
    if pd_on and log["lambda_pd"] is None:
        # the rules' balance: λ_PD = mean ArcFace loss / mean distillation loss over a few batches, before any update
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), 999, ENC_ORDER.index(ek)]); ce_l, pd_l = [], []
        enc.train(); head.train(); arc.train()
        with torch.no_grad():
            for _ in range(CFG["pd_balance_batches"]):
                ids = rng.choice(len(items), CFG["batch"], replace=False)
                X, T, has = make_batch(track, inp, fold, items, ids, rng, n, True)
                y = torch.from_numpy(items[ids, 3]).to(DEVICE)
                ce, pd = step_losses(enc, arc, head, X, y, False, 1.0, None, T, has, n)
                ce_l.append(float(ce)); pd_l.append(float(pd)) if pd is not None else None
        log["lambda_pd"] = round(float(np.mean(ce_l) / max(np.mean(pd_l), 1e-6)), 4)
        print(f"{tag}: λ_PD = {log['lambda_pd']} (ArcFace {np.mean(ce_l):.3f}, distillation {np.mean(pd_l):.4f} at the start)")
    print(f"{tag}: {len(items)} training clips ({log['n_xc']} from DCASE 2020's additional training data) · {len(classes)} classes · {spe} steps per epoch", flush=True)
    for ep in range(start, CFG["epochs"]):
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), ep, ENC_ORDER.index(ek)])
        np.random.seed(seed * 1000 + ep)                              # BEATs draws its layer-drop from numpy's global stream
        order = rng.permutation(len(items))
        enc.train(); head.train(); arc.train()
        t0, tot, tce, tpd, n_mix = time.time(), 0.0, 0.0, 0.0, 0
        for b in range(spe):
            ids = order[b * CFG["batch"]:(b + 1) * CFG["batch"]]
            mix = bool(rng.random() < CFG["mixup_prob"])
            X, T, has = make_batch(track, inp, fold, items, ids, rng, n, pd_on)
            y = torch.from_numpy(items[ids, 3]).to(DEVICE)
            lam, perm = 1.0, None
            if mix:
                lam = float(rng.beta(CFG["mixup_beta"], CFG["mixup_beta"]))
                perm = torch.from_numpy(rng.permutation(len(ids))).to(DEVICE)
                X = lam * X + (1.0 - lam) * X[perm]
                if pd_on:
                    T = lam * T + (1.0 - lam) * T[perm]; has = has & has[perm]
            ce, pd = step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n)
            loss = ce if pd is None else ce + log["lambda_pd"] * pd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, CFG["clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); tce += float(ce.detach()); tpd += 0.0 if pd is None else float(pd.detach()); n_mix += mix
        log["epochs"].append({"epoch": ep, "mean_loss": round(tot / spe, 4), "mean_arcface": round(tce / spe, 4),
                              "mean_distillation": round(tpd / spe, 5), "mixup_batches": int(n_mix), "seconds": round(time.time() - t0, 1)})
        print(f"{tag} · epoch {ep + 1}/{CFG['epochs']} · loss {tot / spe:.3f}" + (f" (distillation {tpd / spe:.4f})" if pd_on else "")
              + f" · {time.time() - t0:.0f} s", flush=True)
        torch.save({**adapter_state(enc, head, arc), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "log": log}, resume)
    enc.eval(); head.eval(); arc.eval()
    torch.save({**adapter_state(enc, head, arc), "classes": classes, "log": log}, final)
    resume.unlink(missing_ok=True)
    return head, log


@torch.no_grad()
def run_pass(enc, mm, head=None, rows=None, views=True):
    # whole clips through the encoder as it stands (LoRA as switched): per clip the global mean P, F = head(P) when a head is
    # given, and the band views: per frequency row, the mean, the max and RDP (γ = 4) over the time patches
    rows = np.arange(mm.shape[0]) if rows is None else rows
    n, bs = mm.shape[1], CFG["embed_batch"]; enc.eval()
    P = np.zeros((len(rows), enc.dim), np.float16); Fh = np.zeros((len(rows), CFG["emb_dim"]), np.float16) if head is not None else None
    V = {v: np.zeros((len(rows), 8, enc.dim), np.float16) for v in VIEWS} if views else None
    for i in range(0, len(rows), bs):
        x = torch.from_numpy(f32(mm[rows[i:i + bs]])).to(DEVICE); m_ = len(x)
        g = enc.grid(enc.tokens(x, amp=AMP), n)
        p = g.mean((1, 2)); P[i:i + m_] = p.cpu().numpy()
        if head is not None:
            Fh[i:i + m_] = head(p).cpu().numpy()
        if views:
            V["mean"][i:i + m_] = g.mean(1).cpu().numpy(); V["max"][i:i + m_] = g.amax(1).cpu().numpy()
            d = (g - g.mean(1, keepdim=True)).norm(dim=-1)            # RDP (arXiv 2603.04605): deviation from the band's mean
            dh = d / d.amax(1, keepdim=True).clamp_min(1e-12)
            w = (1.0 + dh) ** RDP_GAMMA; w = w / w.sum(1, keepdim=True)
            V[f"rdp{RDP_GAMMA}"][i:i + m_] = (w.unsqueeze(-1) * g).sum(1).cpu().numpy()
    return P, Fh, V


LOGS, T_TRAIN = {}, {}
DC_DIR = OUT / "dcase"; DC_DIR.mkdir(exist_ok=True)
ECHO_DC = {}
for t in TYPES:
    f = DC_DIR / f"echo_{t}.npy"
    if not f.exists():
        tmp = DC_DIR / f"echo_{t}.part.npy"; np.save(tmp, echo_stream(DST[t]).astype(np.float16)); os.replace(tmp, f)
    ECHO_DC[t] = np.load(f)
T_ECHO = round(time.time() - t_p1, 1)
print(f"ECHO on every clip: {({t: list(ECHO_DC[t].shape) for t in TYPES})} · {T_ECHO / 60:.1f} min in this session", flush=True)


def dcase_training_set():
    # DCASE 2020's own lists: one class per development machine (all its train/ clips) and per evaluation machine (its first
    # CFG["xc_cap"] additional-training clips by name, as XC); items (type index, source 2 = development / 3 = additional, row, class)
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        nm = np.array(DMETA[t]["name"])
        for set_, src_, cap in (("dev", 2, None), ("eval", 3, CFG["xc_cap"])):
            mids = np.array(DMETA[t]["mid"])
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                rows = rows[:cap] if cap else rows
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020-{'development' if set_ == 'dev' else 'additional'}")
                items.extend((ti, src_, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes


_it, _cl = dcase_training_set()
print(f"DCASE 2020 training lists: {len(_it)} clips, {len(_cl)} classes ({int((_it[:, 1] == 3).sum())} additional-training clips) · "
      f"one training takes about {len(_it) / CFG['batch'] * CFG['epochs'] * ENC['BEATs'].step_s * 1.15 / 60:.0f} min on this GPU")
F_CLASSES, F_TRAIN = list(_cl), int(len(_it))
assert len(F_CLASSES) == REF_CLASSES and F_CLASSES == [f"{t}|{q}|DCASE2020-{w}" for t in TYPES for w, ids in (("development", DEV_IDS[t]), ("additional", EVAL_IDS[t]))
                                                       for q in ids], "stop: the classes are not the rules' 13 machines"
assert all(int((_it[:, 3] == c).sum()) <= FROZEN_CAP for c, nm_ in enumerate(F_CLASSES) if nm_.endswith("additional")), "more than the cap from an evaluation machine"
print(f"the six trainings together: about {6 * len(_it) / CFG['batch'] * CFG['epochs'] * ENC['BEATs'].step_s * 1.15 / 60:.0f} min on this GPU, "
      f"and a pass over {sum(len(DMETA[t]['name']) for t in TYPES)} clips after each", flush=True)


def ensure_dc(ek, seed):
    # one model of MIC's recipe on DCASE's lists (no folds), then F and P of every DCASE clip; gate G1
    tag = n22_tag(ek, "DC", "CH0", ["XC"], seed, 0); fin, ef = OUT / f"ft_{tag}.pt", OUT / f"emb_{tag}.npz"
    if not (fin.exists() and ef.exists()):
        t0 = time.time()
        head, _ = train(ek, "DC", "CH0", ["XC"], seed, 0)
        LoRASwitch.on = True; head.eval(); out = {}
        for t in TYPES:
            P, Fh, _ = run_pass(ENC[ek], DST[t], head, views=False)
            out[f"P__{t}"], out[f"F__{t}"] = P, Fh
        tmp = OUT / f"emb_{tag}.part.npz"
        np.savez(tmp, **out, meta=json.dumps({"tag": tag, "weights_md5": md5(fin), "encoder": ek, "test_run": TEST, "date": time.strftime("%Y-%m-%d %H:%M")}))
        os.replace(tmp, ef)
        T_TRAIN[tag] = round(time.time() - t0, 1)
        print(f"saved {ef.name} · {T_TRAIN[tag] / 60:.1f} min for training and the pass", flush=True)
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    log = torch.load(fin, map_location="cpu", weights_only=False)["log"]; LOGS[tag] = log
    return bool(log["epochs"][-1]["mean_loss"] < log["epochs"][0]["mean_loss"])


# the six trainings of the frozen recipe (finished ones are read from Drive; an unfinished one continues from its last epoch)
DC_G1 = {}
for ek, _, seeds in FROZEN:
    for s in seeds:
        DC_G1[f"{ek}|seed {s}"] = ensure_dc(ek, s)
G1_F = {k: {"first_epoch_loss": LOGS[k]["epochs"][0]["mean_loss"], "last_epoch_loss": LOGS[k]["epochs"][-1]["mean_loss"], "epochs": len(LOGS[k]["epochs"]),
            "seconds": [e["seconds"] for e in LOGS[k]["epochs"]], "resumed_at_epoch": LOGS[k]["resumed_at_epoch"]} for k in LOGS}
print("G1 · the training loss fell in each of the six trainings of the frozen recipe:", all(DC_G1.values()), DC_G1)
if not TEST and not all(DC_G1.values()):
    raise SystemExit("stop: G1 failed for a training of the frozen recipe")
assert all(LOGS[k]["classes"] == F_CLASSES and LOGS[k]["cfg"] == CFG for k in LOGS), "stop: a saved training used other classes or settings"

# Part 1 is on disk (ft_*.pt, emb_*.npz, dcase/echo_*.npy). Its models leave the GPU before the view is trained
ENC_INFO = {k: {**e.info, "name": e.name, "layout": e.layout, "step_seconds": e.step_s} for k, e in ENC.items()}
LOGS_F, T_TRAIN_F = LOGS, T_TRAIN
ENC.clear()
for _n in ("model", "chk", "LORA", "ECHO", "enc", "x4", "Z4", "ref"):
    globals().pop(_n, None)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
T_PART1 = round(time.time() - t_p1, 1)
print(f"Part 1 done · {T_PART1 / 60:.0f} min in this cell · {(time.time() - T_START) / 60:.0f} min in this session", flush=True)

In [ ]:
# 11 · The frozen system from Part 1's saved files (notebook 25's code, reading this notebook's folder): MIC's scorer, and the spread
# σ_F of its log score on normal clips its trainings did not use. No test clip is scored in this cell.
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def dc_metrics(y, s):
    # as the official evaluator: sklearn's AUC, and its pAUC with max_fpr = 0.1 (standardised)
    return {"AUC": float(roc_auc_score(y, s)), "pAUC": float(roc_auc_score(y, s, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}


_missing = [f for f in [f"emb_{c}_DC_XC_s{s}_f0.npz" for _, c, sd in FROZEN for s in sd] + [f"dcase/echo_{t}.npy" for t in TYPES] if not (OUT / f).exists()]
if _missing:
    raise SystemExit(f"stop: Part 1's files are not all in {OUT} (for example {_missing[0]})")
assert all(len(ECHO_DC[t]) == len(DMETA[t]["name"]) for t in TYPES), "stop: ECHO's saved embeddings do not have this session's number of clips"
FM = {}


def frozen_matrix(code, seeds, t):
    k = (code, t)
    if k not in FM:
        Es = [np.load(OUT / f"emb_{code}_DC_XC_s{s}_f0.npz", allow_pickle=False)[f"F__{t}"].astype(np.float32) for s in seeds]
        E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
        FM[k] = np.concatenate([E, unit(ECHO_DC[t].astype(np.float64))], 1) / math.sqrt(2.0)
        assert len(FM[k]) == len(DMETA[t]["name"]), f"Part 1's embeddings of {t} do not have this session's number of clips"
    return FM[k]


def machines(t, set_):
    return sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, set_, "test")].tolist()))


def frozen_score(t, ref, x):
    return sum(ldn1(frozen_matrix(c, sd, t)[ref], frozen_matrix(c, sd, t)[x]) for _, c, sd in FROZEN) / len(FROZEN)


def spread(logs):
    # the centre and the robust spread (1.4826 × the median absolute deviation) of a system's log scores on normal clips
    if len(logs) == 0:
        return {"n": 0, "median": None, "sigma": None}
    med = float(np.median(logs))
    return {"n": int(len(logs)), "median": med, "sigma": float(1.4826 * np.median(np.abs(logs - med)))}


def part_score(code, t, ref, x):
    # one encoder's part of MIC's scorer on its own: its seeds' F with ECHO concatenated (the frozen system is the mean of the two parts)
    sd = next(s for _, c, s in FROZEN if c == code)
    return ldn1(frozen_matrix(code, sd, t)[ref], frozen_matrix(code, sd, t)[x])


def echo_score(t, ref, x):
    # ECHO alone: global LDN1 on its frozen embedding
    E = ECHO_DC[t].astype(np.float64)
    return ldn1(E[ref], E[x])


SF = {}                                                               # (type, set, machine) → the frozen system's scores of its test clips


def frozen_of(t, set_, q):
    # computed once per machine; the references are all of the machine's normal training clips
    k = (t, set_, q)
    if k not in SF:
        SF[k] = frozen_score(t, dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q))
    return SF[k]


_R, _X = np.random.default_rng(11).standard_normal((40, 9)), np.random.default_rng(12).standard_normal((7, 9))
assert np.allclose(ldn1(_R, _X), [min((1 - unit(x) @ unit(r)) / max(1 - max(unit(r) @ unit(r2) for j, r2 in enumerate(_R) if j != i), LDN_FLOOR)
                                      for i, r in enumerate(_R)) for x in _X]), "stop: LDN1 is not the rules' ratio"

t_f = time.time()
LAB = {t: np.array(DMETA[t]["label"]) for t in TYPES}
CAL_F, CALF_SC = {}, {}
for t in TYPES:
    # σ_F (notebook 25's rule): the frozen system's log score on the additional-training clips that its trainings did not use (those
    # after the first FROZEN_CAP of an evaluation machine, by name), against the clips they did use. No test clip.
    _nm, _logs, CALF_SC[t] = np.array(DMETA[t]["name"]), [], []
    for q in machines(t, "eval"):
        rows = dc_rows(t, "eval", "train", q); rows = rows[np.argsort(_nm[rows], kind="stable")]
        if len(rows) > FROZEN_CAP:
            _s = frozen_score(t, rows[:FROZEN_CAP], rows[FROZEN_CAP:])
            _logs.append(np.log(np.maximum(_s, LDN_FLOOR))); CALF_SC[t].extend(zip(_nm[rows[FROZEN_CAP:]].tolist(), _s.tolist()))
    CAL_F[t] = spread(np.concatenate(_logs) if _logs else np.zeros(0))
    FM.clear(); gc.collect()
print("form ZP · the frozen system's spread on normal clips it did not train on (clips, σ of the log score):",
      {t: (CAL_F[t]["n"], None if CAL_F[t]["sigma"] is None else round(CAL_F[t]["sigma"], 4)) for t in TYPES}, f"· {time.time() - t_f:.0f} s")
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True                             # as notebook 25's setup: the view's inputs have one size

In [ ]:
# 12 · Part 2 · The second view (notebook 25's cell): TASTgram-MFN trained with Noisy-ArcMix (Choi and Choi, ICASSP 2024, arXiv 2310.06364)
#
# The model classes, the log-mel settings and the Noisy-ArcMix step are adapted from https://github.com/soonhyeon/Noisy-ArcMix at
# commit 674c759bfaa7723619a4b1d8d00eb9d155c50ba6 (model/net.py, losses.py, dataloader.py, utils.py, trainer.py). Its model file
# says: "modification made on the basis of link:https://github.com/liuyoude/STgram-MFN". Its licence:
#
#   MIT License
#
#   Copyright (c) 2023 soonhyeon
#
#   Permission is hereby granted, free of charge, to any person obtaining a copy
#   of this software and associated documentation files (the "Software"), to deal
#   in the Software without restriction, including without limitation the rights
#   to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
#   copies of the Software, and to permit persons to whom the Software is
#   furnished to do so, subject to the following conditions:
#
#   The above copyright notice and this permission notice shall be included in all
#   copies or substantial portions of the Software.
#
#   THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
#   IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
#   FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
#   AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
#   LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
#   OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
#   SOFTWARE.
#
# Changed here (the rules list every deviation): the number of classes; the frame count and the last kernel follow the clip length
# (313 and 8 × 20 for DCASE's 10 s clips, as in the reference); mixed precision with a gradient scaler, and the ArcFace layer in
# float32 with a floor under its square root; batches come from one int16 tensor and the log-mel is computed on the device; the
# seeds, the hold-out draw, saving and resuming are this notebook's. The reference's unused options (sub-centres, easy margin,
# the other training modes) are left out.
class Bottleneck(nn.Module):
    def __init__(self, inp, oup, stride, expansion):
        super(Bottleneck, self).__init__()
        self.connect = stride == 1 and inp == oup
        self.conv = nn.Sequential(
            # pw
            nn.Conv2d(inp, inp * expansion, 1, 1, 0, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # dw
            nn.Conv2d(inp * expansion, inp * expansion, 3, stride, 1, groups=inp * expansion, bias=False),
            nn.BatchNorm2d(inp * expansion),
            nn.PReLU(inp * expansion),
            # pw-linear
            nn.Conv2d(inp * expansion, oup, 1, 1, 0, bias=False),
            nn.BatchNorm2d(oup),
        )

    def forward(self, x):
        if self.connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class ConvBlock(nn.Module):
    def __init__(self, inp, oup, k, s, p, dw=False, linear=False):
        super(ConvBlock, self).__init__()
        self.linear = linear
        if dw:
            self.conv = nn.Conv2d(inp, oup, k, s, p, groups=inp, bias=False)
        else:
            self.conv = nn.Conv2d(inp, oup, k, s, p, bias=False)
        self.bn = nn.BatchNorm2d(oup)
        if not linear:
            self.prelu = nn.PReLU(oup)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        if self.linear:
            return x
        else:
            return self.prelu(x)


Mobilefacenet_bottleneck_setting = [
    # t, c , n ,s
    [2, 128, 2, 2],
    [4, 128, 2, 2],
    [4, 128, 2, 2],
]


class MobileFaceNet(nn.Module):
    def __init__(self, num_class, last_kernel, bottleneck_setting=Mobilefacenet_bottleneck_setting):
        super(MobileFaceNet, self).__init__()
        self.conv1 = ConvBlock(3, 64, 3, 2, 1)
        self.dw_conv1 = ConvBlock(64, 64, 3, 1, 1, dw=True)
        self.inplanes = 64
        block = Bottleneck
        self.blocks = self._make_layer(block, bottleneck_setting)
        self.conv2 = ConvBlock(bottleneck_setting[-1][1], 512, 1, 1, 0)
        self.linear7 = ConvBlock(512, 512, last_kernel, 1, 0, dw=True, linear=True)   # (8, 20) in the reference: 128 × 313 → 8 × 20
        self.linear1 = ConvBlock(512, 128, 1, 1, 0, linear=True)
        self.fc_out = nn.Linear(128, num_class)                       # the reference has it too; with ArcFace on, its output is unused
        # init
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, setting):
        layers = []
        for t, c, n, s in setting:
            for i in range(n):
                if i == 0:
                    layers.append(block(self.inplanes, c, s, t))
                else:
                    layers.append(block(self.inplanes, c, 1, t))
                self.inplanes = c
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.dw_conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.linear7(x)
        x = self.linear1(x)
        return x.reshape(x.size(0), -1)                               # the 128-d feature


class TgramNet(nn.Module):
    def __init__(self, n_frames, num_layer=3, mel_bins=128, win_len=1024, hop_len=512):
        super(TgramNet, self).__init__()
        # if "center=True" of stft, padding = win_len / 2
        self.conv_extrctor = nn.Conv1d(1, mel_bins, win_len, hop_len, win_len // 2, bias=False)
        self.conv_encoder = nn.Sequential(
            *[nn.Sequential(
                nn.LayerNorm(n_frames),                               # 313 in the reference (10 s)
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv1d(mel_bins, mel_bins, 3, 1, 1, bias=False),
            ) for _ in range(num_layer)])

    def forward(self, x):
        out = self.conv_extrctor(x)
        out = self.conv_encoder(out)
        return out


class Temporal_Attention(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.feature_dim = feature_dim
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 1, 128, 313)
        x = x.squeeze(1)
        x = x.transpose(1, 2)  # (B, 313, 128)
        x1 = self.max_pool(x)  # (B, 313, 1)
        x2 = self.avg_pool(x)  # (B, 313, 1)
        feats = x1 + x2
        feats = feats.repeat(1, 1, self.feature_dim)
        refined_feats = self.sigmoid(feats).transpose(1, 2) * x.transpose(1, 2)
        return refined_feats


class ArcMarginProduct(nn.Module):
    def __init__(self, in_features=128, out_features=200, s=30.0, m=0.7):
        super(ArcMarginProduct, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = Parameter(torch.Tensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        # make the function cos(theta+m) monotonic decreasing while theta in [0°,180°]
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, x, label):
        cosine = F.linear(F.normalize(x), F.normalize(self.weight))
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp_min(SV["sine_floor"]))   # the floor is this notebook's (a deviation)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where((cosine - self.th) > 0, phi, cosine - self.mm)
        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output = output * self.s
        return output


MEMFMT = torch.contiguous_format                                      # set by pick_memory_format (a speed setting only)


class TASTgramMFN(nn.Module):
    def __init__(self, num_classes, n_frames, last_kernel, c_dim=128, win_len=1024, hop_len=512,
                 bottleneck_setting=Mobilefacenet_bottleneck_setting, m=0.7, s=30):
        super().__init__()
        self.arcface = ArcMarginProduct(in_features=c_dim, out_features=num_classes, m=m, s=s)
        self.tgramnet = TgramNet(n_frames, mel_bins=c_dim, win_len=win_len, hop_len=hop_len)
        self.mobilefacenet = MobileFaceNet(num_class=num_classes, last_kernel=last_kernel, bottleneck_setting=bottleneck_setting)
        self.temporal_attention = Temporal_Attention(feature_dim=c_dim)

    def forward(self, x_wav, x_mel, label):
        # the reference's "noisy_arcmix" mode: the margin goes to `label` (in training, the first clip's class)
        with torch.autocast(device_type=DEVICE, enabled=AMP):
            x_t = self.tgramnet(x_wav).unsqueeze(1)
            x_mel_temp_att = self.temporal_attention(x_mel).unsqueeze(1)
            x = torch.cat((x_t.float(), x_mel, x_mel_temp_att), dim=1)   # .float(): the three channels in one type under autocast
            feature = self.mobilefacenet(x.contiguous(memory_format=MEMFMT))
        feature = feature.float()
        with torch.autocast(device_type=DEVICE, enabled=False):      # ArcFace in float32 (a deviation)
            out = self.arcface(feature, label)
        return out, feature


MEL = torchaudio.transforms.MelSpectrogram(sample_rate=SR, n_fft=SV["n_fft"], hop_length=SV["hop"], n_mels=SV["n_mels"], power=2,
                                           pad_mode="constant", norm="slaney", mel_scale="slaney").to(DEVICE)
CE = nn.CrossEntropyLoss()


@torch.no_grad()
def log_mel(x):
    # (B, samples) float32 in [−1, 1] → (B, 1, mel bands, frames): the reference's file_to_log_mel_spectrogram with power = 2
    return (20.0 / 2 * torch.log10(MEL(x) + sys.float_info.epsilon)).unsqueeze(1)


def frames_of(ns):
    # the frames of an STFT with center=True, which is also the length of the Tgram convolution (kernel 1024, stride 512, padding 512)
    return 1 + ns // SV["hop"]


def last_kernel(n_mels, n_frames):
    # the feature map after the network's four stride-2 stages (3 × 3, padding 1): 128 × 313 → 8 × 20
    h, w = n_mels, n_frames
    for _ in range(4):
        h, w = (h - 1) // 2 + 1, (w - 1) // 2 + 1
    return (h, w)


def training_lists():
    # one class per machine: for each type, its development machines (all train/ clips of the development zip), then its evaluation
    # machines (all clips of the additional training zip); items (type index, row in the type's store, class)
    items, classes, cls_of = [], [], {}
    for ti, t in enumerate(TYPES):
        nm, mids = np.array(DMETA[t]["name"]), np.array(DMETA[t]["mid"])
        for set_, what in (("dev", "development"), ("eval", "additional")):
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020-{what}"); cls_of[(t, q)] = c
                items.extend((ti, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes, cls_of


AUDIO, LABELS, STORE_DEV, MF_TIMES = None, None, None, {}


def build_audio():
    # every training-list clip in one int16 tensor: on the GPU when it has room (batches are then gathered without the CPU), else in RAM
    global AUDIO, LABELS, STORE_DEV
    if AUDIO is not None:
        return
    n = len(ITEMS); need = n * NS * 2 / 1e9; dev = DEVICE; t0 = time.time()
    if DEVICE == "cuda" and torch.cuda.mem_get_info()[0] / 1e9 < need + SV["gpu_headroom_gb"]:
        dev = "cpu"
    A = torch.empty((n, NS), dtype=torch.int16, device=dev)
    for ti, t in enumerate(TYPES):
        pos = np.where(ITEMS[:, 0] == ti)[0]
        for i in range(0, len(pos), 256):
            p = pos[i:i + 256]
            A[torch.from_numpy(p).to(dev)] = torch.from_numpy(np.ascontiguousarray(DST[t][ITEMS[p, 1]])).to(dev)
    AUDIO, STORE_DEV, LABELS = A, dev, torch.from_numpy(ITEMS[:, 2].copy()).to(dev)
    print(f"training audio: {n} clips, {need:.2f} GB as int16, kept on {'the GPU' if dev == 'cuda' else 'the CPU'} · {time.time() - t0:.0f} s", flush=True)


def fetch(ids):
    # clips `ids` of the training lists → float32 waves in [−1, 1] and their classes, on the device
    j = torch.from_numpy(np.ascontiguousarray(ids)).to(AUDIO.device)
    return AUDIO[j].to(DEVICE).float() / 32768.0, LABELS[j].to(DEVICE)


def build_net(seed):
    torch.manual_seed(seed)
    return TASTgramMFN(len(CLASSES), N_FRAMES, LAST_KERNEL, c_dim=SV["c_dim"], win_len=SV["win_len"], hop_len=SV["hop"], m=SV["arc_m"], s=SV["arc_s"])


def mixed_step(net, x, y, lam, perm):
    # the reference's mixup_data and noisy_arcmix_criterion: the waves and the log-mels are mixed with one λ and one permutation;
    # the margin is at the first clip's class; the loss is λ·CE(first class) + (1 − λ)·CE(second class) on the same logits
    mel = log_mel(x)
    logits, _ = net((lam * x + (1 - lam) * x[perm]).unsqueeze(1), lam * mel + (1 - lam) * mel[perm], y)
    return lam * CE(logits, y) + (1 - lam) * CE(logits, y[perm]), logits


def pick_memory_format():
    # the same mathematics in two memory layouts: a few training steps are timed in each on this GPU, and the faster one is kept.
    # These are also the first real training steps: if neither layout runs with the audio on the GPU (most likely the GPU's memory
    # is too small for both), the audio is moved to the CPU's memory and the steps are tried once more.
    global MEMFMT, AUDIO, LABELS, STORE_DEV
    if DEVICE != "cuda":
        return
    err = ""
    for attempt in (0, 1):
        MF_TIMES.clear()
        x, y = fetch(np.arange(min(SV["batch"], len(ITEMS)))); perm = torch.arange(len(y) - 1, -1, -1, device=DEVICE)
        for name, fmt in (("contiguous", torch.contiguous_format), ("channels_last", torch.channels_last)):
            MEMFMT = fmt; net = opt = scaler = loss = None
            try:
                net = build_net(0).to(DEVICE).to(memory_format=fmt); net.train()
                opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"]); scaler = torch.amp.GradScaler("cuda", enabled=AMP)
                for i in range(16):
                    if i == 4:
                        torch.cuda.synchronize(); t0 = time.time()
                    loss, _ = mixed_step(net, x, y, 0.5, perm)
                    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                torch.cuda.synchronize(); MF_TIMES[name] = round((time.time() - t0) / 12, 4)
            except Exception as e:
                MF_TIMES[name] = None; err = repr(e)[:160]
                print(f"  ({name} could not be timed: {err})", flush=True)
            del net, opt, scaler, loss
            gc.collect(); torch.cuda.empty_cache()
        ok = {k: v for k, v in MF_TIMES.items() if v}
        del x, y, perm
        if ok or STORE_DEV != "cuda":
            break
        AUDIO, LABELS, STORE_DEV = AUDIO.cpu(), LABELS.cpu(), "cpu"
        gc.collect(); torch.cuda.empty_cache()
        print("  no training step ran with the audio on the GPU: the audio is moved to the CPU's memory, and the steps are tried again", flush=True)
    if not ok:
        raise SystemExit(f"stop: a training step does not run on this GPU ({err}). Choose G4 in Runtime › Change runtime type, then Run all.")
    best = min(ok, key=ok.get)
    MEMFMT = torch.channels_last if best == "channels_last" else torch.contiguous_format
    print(f"memory layout: seconds per training step {MF_TIMES} → {best}", flush=True)


@torch.no_grad()
def hold_out_pass(net, hold):
    # as the reference's valid(): evaluation mode, margin at the clip's class, no mixing; the mean of the batches' means
    net.eval(); losses, accs = [], []
    for i in range(0, len(hold), SV["batch"]):
        x, y = fetch(hold[i:i + SV["batch"]])
        logits, _ = net(x.unsqueeze(1), log_mel(x), y)
        losses.append(float(CE(logits, y))); accs.append(float((logits.argmax(1) == y).float().mean()))
    return float(np.mean(losses)), float(np.mean(accs))


def fmt_time(s):
    return f"{s / 3600:.1f} h" if s >= 5400 else f"{s / 60:.0f} min"


def hold_out_split():
    # 5 % held out at random, one draw for the three seeds (as notebook 25): the held-out clips are then unseen by every seed's model.
    # The draw is saved in Drive the first time and read from there afterwards, so that a later session uses the same clips.
    n = len(ITEMS); n_tr = int(n * SV["split_ratio"]); f = OUT / "hold_out.json"
    if f.exists():
        d = json.loads(f.read_text())
        assert d["n_clips"] == n and len(d["hold_out"]) == n - n_tr, "the saved hold-out belongs to other training lists"
        hold = np.array(d["hold_out"], dtype=np.int64)
    else:
        hold = np.sort(np.random.default_rng([27, 0]).permutation(n)[n_tr:])
        f.write_text(json.dumps({"n_clips": int(n), "hold_out": hold.tolist()}))
    return np.setdiff1d(np.arange(n), hold), hold


def train_sv(seed):
    # one training of the second view: 5 % held out, Noisy-ArcMix on every batch, AdamW with cosine annealing per epoch, the model
    # with the lowest hold-out loss kept (as the reference); saved to Drive every SV["save_every"] epochs and resumed from there
    fin, res = OUT / f"sv_s{seed}.pt", OUT / f"sv_s{seed}_resume.pt"
    build_audio()
    tr_idx, hold = hold_out_split(); n, n_tr = len(ITEMS), len(tr_idx)
    hold_md5 = hashlib.md5(hold.astype("<i8").tobytes()).hexdigest()
    bs, epochs = SV["batch"], SV["epochs"]; steps = math.ceil(n_tr / bs)
    net = build_net(seed).to(DEVICE).to(memory_format=MEMFMT)
    opt = torch.optim.AdamW(net.parameters(), lr=SV["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=SV["eta_min_ratio"] * SV["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"seed": seed, "cfg": dict(SV), "classes": CLASSES, "n_clips": int(n), "n_train": int(n_tr), "n_hold_out": int(n - n_tr),
           "hold_out_md5": hold_md5, "steps_per_epoch": int(steps), "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL),
           "parameters": int(sum(p.numel() for p in net.parameters())), "epochs": [], "resumed_at_epoch": [], "sessions": []}
    best, start = {"val_loss": float("inf"), "epoch": -1, "state": None}, 0
    if res.exists():
        st = torch.load(res, map_location="cpu", weights_only=True)
        log = json.loads(st["log"])
        assert log["classes"] == CLASSES and log["hold_out_md5"] == hold_md5 and log["cfg"] == json.loads(json.dumps(SV)), \
            "the saved training used other lists or settings"
        net.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        best = {"val_loss": st["best_val_loss"], "epoch": st["best_epoch"], "state": st["best_state"]}
        start = st["epoch"] + 1; log["resumed_at_epoch"].append(start)
        print(f"seed {seed}: resuming after epoch {start}", flush=True)
    session = {"gpu": GPU, "memory_layout": "channels_last" if MEMFMT == torch.channels_last else "contiguous", "audio_on": STORE_DEV,
               "from_epoch": start, "step_seconds": None}
    log["sessions"].append(session)
    print(f"seed {seed}: {n_tr} training clips and {n - n_tr} held out · {len(CLASSES)} classes · {steps} steps per epoch · "
          f"{log['parameters']:,} parameters", flush=True)
    t_from, t_to = min(5, steps - 2), min(45, steps - 1)
    for ep in range(start, epochs):
        rng = np.random.default_rng([seed, 27, 1, ep]); order = tr_idx[rng.permutation(n_tr)]
        lr_now = opt.param_groups[0]["lr"]
        net.train(); t0, tot, acc = time.time(), 0.0, 0.0
        for b in range(steps):
            x, y = fetch(order[b * bs:(b + 1) * bs])
            lam = float(rng.beta(SV["alpha"], SV["alpha"])); pb = torch.from_numpy(rng.permutation(len(y))).to(DEVICE)
            loss, logits = mixed_step(net, x, y, lam, pb)
            if not bool(torch.isfinite(loss)):
                raise SystemExit(f"stop: G1 failed. The training loss of seed {seed} is not finite (epoch {ep + 1}, step {b + 1})")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += float(loss.detach()); acc += float((logits.detach().argmax(1) == y).float().mean())
            if ep == start and b in (t_from, t_to) and t_to > t_from >= 0:
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                if b == t_from:
                    t_mark = time.time()
                else:
                    session["step_seconds"] = round((time.time() - t_mark) / (t_to - t_from), 4)
                    left_here = (epochs - ep) * steps * session["step_seconds"] * 1.03
                    later = [s for s in SV["seeds"] if s != seed and not (OUT / f"sv_s{s}.pt").exists()]
                    left_all = left_here + len(later) * epochs * steps * session["step_seconds"] * 1.03
                    print(f"  this GPU ({GPU or 'CPU'}): {session['step_seconds']:.3f} s per step → about {fmt_time(left_here)} left for this "
                          f"training, about {fmt_time(left_all)} for all the trainings that are left", flush=True)
                    if left_all > 6 * 3600 and not TEST:
                        print("  NOTE: that is a long run on this GPU. It can stay as it is (it resumes after a disconnect). For a much shorter "
                              "run: Runtime › Change runtime type › G4, then Runtime › Run all; what is saved in Drive is kept.", flush=True)
        sched.step()
        vl, va = hold_out_pass(net, hold)
        if not math.isfinite(vl):
            raise SystemExit(f"stop: G1 failed. The hold-out loss of seed {seed} is not finite (epoch {ep + 1})")
        if best["val_loss"] > vl:                                     # the reference saves the model whenever the hold-out loss falls
            best = {"val_loss": vl, "epoch": ep, "state": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}}
        log["epochs"].append({"epoch": ep, "train_loss": round(tot / steps, 5), "train_accuracy": round(acc / steps, 5), "hold_out_loss": round(vl, 5),
                              "hold_out_accuracy": round(va, 5), "lr": lr_now, "seconds": round(time.time() - t0, 1)})
        if ep < start + 3 or (ep + 1) % 5 == 0 or ep + 1 == epochs:
            print(f"seed {seed} · epoch {ep + 1}/{epochs} · loss {tot / steps:.4f} · hold-out loss {vl:.4f}, accuracy {va:.4f} · "
                  f"best epoch {best['epoch'] + 1} · {time.time() - t0:.0f} s", flush=True)
        if (ep + 1) % SV["save_every"] == 0 and ep + 1 < epochs:
            tmp = res.with_name(res.name + ".tmp")
            torch.save({"model": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": ep, "best_state": best["state"], "best_epoch": best["epoch"],
                        "best_val_loss": best["val_loss"], "log": json.dumps(log)}, tmp)
            os.replace(tmp, res)
    log["best_epoch"], log["best_hold_out_loss"] = int(best["epoch"]), float(best["val_loss"])
    tmp = fin.with_name(fin.name + ".tmp")
    torch.save({"state": best["state"], "last_state": {k: v.detach().cpu() for k, v in net.state_dict().items()}, "best_epoch": int(best["epoch"]),
                "best_val_loss": float(best["val_loss"]), "classes": CLASSES, "hold_out": hold.tolist(), "log": json.dumps(log)}, tmp)
    os.replace(tmp, fin)
    res.unlink(missing_ok=True)
    del net, opt, sched, scaler
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


@torch.no_grad()
def embed_sv(seed):
    # the saved model on every DCASE clip: the 128-d feature, and the paper's score (cross-entropy for the clip's own machine class)
    fin = OUT / f"sv_s{seed}.pt"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["classes"] == CLASSES, "the saved model used other classes"
    net = build_net(seed); net.load_state_dict(st["state"]); net.to(DEVICE).to(memory_format=MEMFMT).eval()
    out, bs = {}, SV["embed_batch"]
    for t in TYPES:
        y_all = np.array([CLS_OF[(t, q)] for q in DMETA[t]["mid"]], dtype=np.int64); n = DST[t].shape[0]
        V, ce = np.zeros((n, SV["c_dim"]), np.float32), np.zeros(n, np.float32)
        for i in range(0, n, bs):
            x = torch.from_numpy(f32(DST[t][i:i + bs])).to(DEVICE); y = torch.from_numpy(y_all[i:i + bs]).to(DEVICE)
            logits, feat = net(x.unsqueeze(1), log_mel(x), y)
            V[i:i + len(y)] = feat.cpu().numpy(); ce[i:i + len(y)] = F.cross_entropy(logits, y, reduction="none").cpu().numpy()
        assert np.isfinite(V).all() and np.isfinite(ce).all(), f"non-finite features for {t}"
        out[f"V__{t}"], out[f"CE__{t}"] = V, ce
    tmp = OUT / f"emb_SV_s{seed}.part.npz"
    np.savez(tmp, **out, meta=json.dumps({"seed": seed, "weights_md5": md5(fin), "best_epoch": int(st["best_epoch"]), "test_run": TEST,
                                          "date": time.strftime("%Y-%m-%d %H:%M")}))
    os.replace(tmp, OUT / f"emb_SV_s{seed}.npz")
    del net
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 13 · How systems are scored on a machine's test clips: the kept system (form ZP), its two parts, and the reported readings
VIEWS, CES, VM = {}, {}, {}                                           # seed → type → features (n, 128) and the paper's score (n,)
HOLD_ROWS, CAL_P = {}, {}                                             # type → store rows of the held-out clips; (seeds, type) → the view's spread
SYSTEMS = ["ZP", "frozen", "view_paper", "view_LDN1", "view_B1", "frozen_B", "frozen_B3", "echo"]
LABEL = {"ZP": "kept system · frozen + view (ZP)", "frozen": "frozen recipe alone", "view_paper": "view alone · paper's score",
         "view_LDN1": "view alone · LDN1", "view_B1": "view alone · B1", "frozen_B": "frozen · BEATs iter3+ part",
         "frozen_B3": "frozen · BEATs iter3 part", "echo": "ECHO alone · LDN1"}


def view_matrix(seeds, t):
    # the program's seed ensemble: each seed's features unit-normalised, concatenated, ÷√seeds
    k = (tuple(seeds), t)
    if k not in VM:
        VM[k] = np.concatenate([unit(VIEWS[s][t]) for s in seeds], 1) / math.sqrt(len(seeds))
    return VM[k]


def calibrate_paper(seeds, t):
    # form ZP's calibration of the view: the log of the paper's score (averaged over the seeds) on the held-out clips. No test clip.
    k = (tuple(seeds), t)
    if k not in CAL_P:
        rows = np.asarray(HOLD_ROWS[t])
        CAL_P[k] = spread(np.log(np.maximum(sum(CES[s][t][rows] for s in seeds) / len(seeds), LDN_FLOOR)) if len(rows) else np.zeros(0))
    return CAL_P[k]


def paper(seeds, t, rows):
    # the paper's score (the cross-entropy for the clip's own machine class, margin on), averaged over the seeds
    return sum(CES[s][t][rows] for s in seeds) / len(seeds)


def zp_ok(seeds):
    # form ZP needs a spread from at least CAL_MIN clips, for both systems and every type
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (CAL_F[t], calibrate_paper(seeds, t)))


def score_of(name, t, set_, q, seeds):
    # one system's anomaly scores of one machine's test clips; the references are all of the machine's normal training clips
    ref, te = dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q)
    if name == "frozen":
        return frozen_of(t, set_, q)
    if name == "view_paper":
        return paper(seeds, t, te)
    if name == "ZP":                                                  # each system's log score in units of its spread on unseen normal clips
        return (np.log(np.maximum(frozen_of(t, set_, q), LDN_FLOOR)) / CAL_F[t]["sigma"]
                + np.log(np.maximum(paper(seeds, t, te), LDN_FLOOR)) / calibrate_paper(seeds, t)["sigma"])
    if name in ("frozen_B", "frozen_B3"):
        return part_score(name.split("_")[1], t, ref, te)
    if name == "echo":
        return echo_score(t, ref, te)
    V = view_matrix(seeds, t)
    if name == "view_LDN1":
        return ldn1(V[ref], V[te])
    if name == "view_B1":
        return b1(V[ref], V[te])
    raise ValueError(name)


def dev_eval(seeds, names, keep=None):
    # the development set. Per system and type: AUC and pAUC per machine and their means (the official metric), the worst machine,
    # and the AUC with the type's machines pooled (what the reference's script computes). With `keep`, every clip's scores are kept too.
    names = [nm for nm in names if nm != "ZP" or zp_ok(seeds)]            # the calibrated form only when its spreads can be formed
    out = {nm: {} for nm in names}
    for t in TYPES:
        qs = machines(t, "dev"); ys = {q: LAB[t][dc_rows(t, "dev", "test", q)] for q in qs}
        for nm in names:
            sc = {q: score_of(nm, t, "dev", q, seeds) for q in qs}
            if keep is not None:
                for q in qs:
                    keep.setdefault((t, q), {"file": [DMETA[t]["name"][i] for i in dc_rows(t, "dev", "test", q)], "label": ys[q]})[nm] = sc[q]
            per = {q: dc_metrics(ys[q], sc[q]) for q in qs}
            y_all, s_all = np.concatenate([ys[q] for q in qs]), np.concatenate([sc[q] for q in qs])
            out[nm][t] = {"per_machine": per, "AUC": float(np.mean([v["AUC"] for v in per.values()])),
                          "pAUC": float(np.mean([v["pAUC"] for v in per.values()])),
                          "worst_machine_AUC": float(min(v["AUC"] for v in per.values())),
                          "pooled_AUC": float(roc_auc_score(y_all, s_all)), "pooled_pAUC": float(roc_auc_score(y_all, s_all, max_fpr=0.1))}
        FM.clear(); VM.clear(); gc.collect()
    return out


def fusion_word(kept, frozen, view):
    # the rules' three words for the fusion, on AUCs of four decimals: against the better of its two parts
    d = round(round(kept, 4) - max(round(frozen, 4), round(view, 4)), 4)
    return ("helps" if d >= Q2_BAR else "costs" if d < -Q2_BAR else "no clear difference"), d


def fusion_table(res):
    # per type: the kept system's AUC less the frozen recipe's and less the view's, and the word
    out = {}
    for t in TYPES:
        k, f, v = (res[nm][t]["AUC"] for nm in ("ZP", "frozen", "view_paper")); w, d = fusion_word(k, f, v)
        out[t] = {"kept": round(k, 4), "frozen": round(f, 4), "view": round(v, 4), "kept_less_frozen": round(round(k, 4) - round(f, 4), 4),
                  "kept_less_view": round(round(k, 4) - round(v, 4), 4), "kept_less_better_part": d, "word": w}
    return out


assert [fusion_word(0.95, 0.94, 0.90)[0], fusion_word(0.9499, 0.94, 0.90)[0], fusion_word(0.93, 0.94, 0.90)[0], fusion_word(0.9299, 0.94, 0.90)[0]] == \
    ["helps", "no clear difference", "no clear difference", "costs"], "stop: the fusion's words are not the rules'"


def show(res, names, title, key="AUC"):
    print(title)
    print(f"  {'':<36}" + "".join(f"{NAME[t]:>14}" for t in TYPES))
    for nm in [n_ for n_ in names if n_ in res]:
        print(f"  {LABEL[nm]:<36}" + "".join(f"{res[nm][t][key]:>14.4f}" for t in TYPES))

In [ ]:
# 14 · Part 2 · The view's three seeds (each: training or resume, then its features of every clip); gates G0(d) and G1. No test clip
# is scored in this cell: the development set is read only after the nine trainings, as the rules order it.
t_p2 = time.time()
ITEMS, CLASSES, CLS_OF = training_lists()
N_FRAMES = frames_of(NS); LAST_KERNEL = last_kernel(SV["n_mels"], N_FRAMES)
_, HOLD = hold_out_split(); N_TRAIN = len(ITEMS) - len(HOLD)
HOLD_ROWS.update({t: ITEMS[HOLD][ITEMS[HOLD][:, 0] == ti][:, 1] for ti, t in enumerate(TYPES)})
print(f"the view's training lists: {len(ITEMS)} clips ({int(sum(1 for c in CLASSES if c.endswith('additional')))} additional-training machines among "
      f"{len(CLASSES)} classes) · {N_FRAMES} frames · last kernel {LAST_KERNEL} · {N_TRAIN} train, {len(HOLD)} held out (the same clips for every seed)")
assert CLASSES == F_CLASSES, "stop: the view's classes are not the frozen recipe's 13 machines"
G0["d"] = {"ok": bool((N_FRAMES, LAST_KERNEL, len(CLASSES)) == (REF_FRAMES, REF_KERNEL, REF_CLASSES)), "frames": int(N_FRAMES),
           "last_kernel": list(LAST_KERNEL), "classes": len(CLASSES), "clips": int(len(ITEMS)), "train": int(N_TRAIN), "held_out": int(len(HOLD)),
           "held_out_per_type": {t: int(len(HOLD_ROWS[t])) for t in TYPES}}
print("G0(d) · the view has the reference's sizes (313 frames, a last kernel of 8 × 20) and 13 classes:", G0["d"]["ok"])
if not TEST and not G0["d"]["ok"]:
    raise SystemExit("stop: G0(d) failed, the lists or the sizes are not the rules'")
_todo = [s for s in SV["seeds"] if not (OUT / f"sv_s{s}.pt").exists()]
print(f"trainings to do: {len(_todo)} of {len(SV['seeds'])} · {math.ceil(N_TRAIN / SV['batch'])} steps per epoch, {SV['epochs']} epochs each. "
      f"At the 0.019 s per step notebook 26 measured on an RTX PRO 6000 (G4), that is about "
      f"{math.ceil(N_TRAIN / SV['batch']) * SV['epochs'] * 0.019 * 1.15 / 60:.0f} min per training there. The measured estimate for this GPU follows.")
if _todo:
    build_audio(); pick_memory_format()
LOGS, G1, T_TRAIN, T_EMBED = {}, {}, {}, {}


def ensure_sv(seed):
    # one seed: trained (or resumed, or read from Drive), then its features of every DCASE clip; gate G1
    fin, ef = OUT / f"sv_s{seed}.pt", OUT / f"emb_SV_s{seed}.npz"
    if not fin.exists():
        t0 = time.time(); train_sv(seed); T_TRAIN[seed] = round(time.time() - t0, 1)
    if ef.exists() and json.loads(str(np.load(ef, allow_pickle=False)["meta"]))["weights_md5"] != md5(fin):
        ef.unlink()                                                   # features of another model file: computed again
    if not ef.exists():
        t0 = time.time(); embed_sv(seed); T_EMBED[seed] = round(time.time() - t0, 1)
        print(f"seed {seed}: features of every DCASE clip saved · {T_EMBED[seed]:.0f} s", flush=True)
    D = np.load(ef, allow_pickle=False)
    VIEWS[seed] = {t: D[f"V__{t}"].astype(np.float64) for t in TYPES}; CES[seed] = {t: D[f"CE__{t}"].astype(np.float64) for t in TYPES}
    assert all(len(VIEWS[seed][t]) == len(DMETA[t]["name"]) for t in TYPES), "the saved features do not have this session's number of clips"
    st = torch.load(fin, map_location="cpu", weights_only=True)
    assert st["hold_out"] == HOLD.tolist(), "the saved model was trained with another hold-out"
    log = json.loads(st["log"]); LOGS[seed] = log
    e = log["epochs"]
    G1[seed] = {"first_train_loss": e[0]["train_loss"], "last_train_loss": e[-1]["train_loss"], "first_hold_out_loss": e[0]["hold_out_loss"],
                "best_hold_out_loss": round(log["best_hold_out_loss"], 5), "best_epoch": log["best_epoch"] + 1,
                "hold_out_accuracy_at_best": e[log["best_epoch"]]["hold_out_accuracy"], "epochs": len(e),
                "ok": bool(len(e) == SV["epochs"] and e[-1]["train_loss"] < e[0]["train_loss"] and log["best_hold_out_loss"] < e[0]["hold_out_loss"])}
    print(f"G1 · seed {seed}: training loss {e[0]['train_loss']:.4f} → {e[-1]['train_loss']:.4f}; hold-out loss {e[0]['hold_out_loss']:.4f} → "
          f"{log['best_hold_out_loss']:.4f} at epoch {log['best_epoch'] + 1} (accuracy {G1[seed]['hold_out_accuracy_at_best']:.4f}): {G1[seed]['ok']}", flush=True)
    if not G1[seed]["ok"] and not TEST:
        raise SystemExit(f"stop: G1 failed for seed {seed}")


for _seed in SV["seeds"]:
    ensure_sv(_seed)
assert all(LOGS[s]["cfg"] == json.loads(json.dumps(SV)) and LOGS[s]["classes"] == CLASSES for s in SV["seeds"]), "stop: a saved view was trained with other settings"
AUDIO = LABELS = None
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
T_PART2 = round(time.time() - t_p2, 1)
print(f"the view's trainings are done · {T_PART2 / 60:.0f} min in this cell · {(time.time() - T_START) / 60:.0f} min in this session")

In [ ]:
# 15 · The development set, read here for the first time. It selects nothing: the kept system is form ZP with both recipes unchanged.
# The spreads of form ZP, the tables (reported, no decision), and gate G-DEV before the evaluation set is touched.
SEEDS = list(SV["seeds"])
CAL = {"frozen": CAL_F, "view_paper": {t: calibrate_paper(SEEDS, t) for t in TYPES}, "frozen_cap": FROZEN_CAP, "minimum_clips": CAL_MIN}
_cs = lambda d: {t: (d[t]["n"], None if d[t]["sigma"] is None else round(d[t]["sigma"], 4)) for t in TYPES}
print("form ZP · the spreads of the log score on normal clips a system did not train on (clips, σ): the frozen recipe", _cs(CAL_F),
      "· the view by the paper's score", _cs(CAL["view_paper"]))
if not zp_ok(SEEDS):
    raise SystemExit(f"stop: a spread of form ZP cannot be formed (it needs at least {CAL_MIN} clips per type and must not be zero): {CAL}")
DEV_SC = {}                                                           # (type, machine) → every development test clip's scores, saved in the zip
DEV = dev_eval(SEEDS, SYSTEMS, keep=DEV_SC)
assert list(DEV) == SYSTEMS
show(DEV, SYSTEMS, f"\nDevelopment set · AUC per type (mean over a type's machines), the view with {len(SEEDS)} seeds:")
show(DEV, SYSTEMS, "\nDevelopment set · pAUC (p = 0.1):", key="pAUC")
print("\nDevelopment set · AUC per machine:")
for t in TYPES:
    for nm in GDEV_SYSTEMS:
        print(f"  {NAME[t]:<12} {LABEL[nm]:<36}" + " ".join(f"{q} {v['AUC']:.4f}" for q, v in DEV[nm][t]["per_machine"].items()))
print("\nPublished development numbers, AUC / pAUC % (all six types trained together; yardsticks, not bars):")
for k_, v_ in PUBLISHED_DEV.items():
    print(f"  {k_}: " + " · ".join(f"{NAME[t]} {v_[t][0]:.2f} / {v_[t][1]:.2f}" for t in TYPES))
print("  the view here, by the paper's score, with a type's machines pooled as the reference's script computes it (AUC %):",
      {NAME[t]: round(100 * DEV["view_paper"][t]["pooled_AUC"], 2) for t in TYPES})
print("  the mean of AUC and pAUC, %:", {NAME[t]: {"kept system": round(50 * (DEV["ZP"][t]["AUC"] + DEV["ZP"][t]["pAUC"]), 2),
                                                   "frozen recipe": round(50 * (DEV["frozen"][t]["AUC"] + DEV["frozen"][t]["pAUC"]), 2),
                                                   "AnoPatch with BEATs (arXiv 2406.11364)": ANOPATCH["development"][t]} for t in TYPES})
PER_SEED = {s: dev_eval([s], ["ZP", "view_paper", "view_LDN1", "view_B1"]) for s in SEEDS}
print("\nSeed spread · development AUC with each seed of the view alone:",
      {s: {NAME[t]: {"kept system": round(PER_SEED[s]["ZP"][t]["AUC"], 4) if "ZP" in PER_SEED[s] else None,
                     "view": round(PER_SEED[s]["view_paper"][t]["AUC"], 4)} for t in TYPES} for s in SEEDS})
FUSION_DEV = fusion_table(DEV)
print("\nThe fusion on the development set (the kept system less the better of its two parts; reported, no decision):",
      {NAME[t]: (FUSION_DEV[t]["kept_less_better_part"], FUSION_DEV[t]["word"]) for t in TYPES})

# gate G-DEV, before the evaluation set is touched: it is there to catch a broken pipeline, not to judge the system
GDEV = {nm: {t: {"AUC": round(DEV[nm][t]["AUC"], 4), "bar": GDEV_BAR[t], "ok": bool(round(DEV[nm][t]["AUC"], 4) >= GDEV_BAR[t])} for t in TYPES}
        for nm in GDEV_SYSTEMS}
GDEV_OK = bool(all(v["ok"] for d in GDEV.values() for v in d.values()))
print(f"\nG-DEV · the kept system and its two parts each have a development AUC of at least {GDEV_BAR['ToyCar']} on ToyCar and "
      f"{GDEV_BAR['ToyConveyor']} on ToyConveyor:", GDEV_OK, GDEV)
if not GDEV_OK:
    raise SystemExit("stop: G-DEV failed; something is broken, and the evaluation set is not scored")

In [ ]:
# 16 · The evaluation set, scored once. Three sets of scores are written before any label is read: the kept system's (the result) and
# each of its two parts' (for question 2). The labels are read only here; a later run reads the saved results and scores nothing
# again. Then the official evaluator on each set, and the readings the rules fixed.
EV_DIR = OUT / "dcase2020_evaluation_toy_types"
EV_RES, EV_LOCK, EV_LABELS = EV_DIR / "results.json", EV_DIR / "scores_written.json", EV_DIR / "eval_data_list_toy_types.csv"
EV_SETS = {"kept": "ZP", "frozen": "frozen", "view": "view_paper"}    # folder → system; "kept" is the result
EV_DIR.mkdir(exist_ok=True)


def read_labels():
    # the ground truth of the two types: (type, file) → label, and the file's lines of these two sections, unchanged
    lab, lines, types_in_file, cur = {}, [], [], None
    with open(get_file(DC_GT), newline="") as fh:                     # stops unless the file's MD5 is the Zenodo record's
        for line in fh:
            row = next(csv.reader([line]), [])
            if not row:
                continue
            if len(row) < 2:                                          # a machine-type line, as the official evaluator reads it
                cur = row[0].strip(); types_in_file.append(cur)
            elif cur in TYPES:
                lab[(cur, row[0].strip())] = int(float(row[2]))
            if cur in TYPES:
                lines.append(line)
    return lab, lines, types_in_file


def official_check(k):
    # the official evaluator (the pinned file, unmodified) on one set of score files, compared with this notebook's numbers
    out = {"ran": False, "evaluator": {"url": EVALUATOR[0], "sha256": EVALUATOR[2]}, "labels": list(DC_GT[1:])}
    src = EV_DIR / k
    files = json.loads(EV_LOCK.read_text())["files"][k]
    if any(not (src / f).exists() for f in files) or not EV_RES.exists():
        out["reason"] = "a score file or results.json of this notebook is missing"
        return out
    wd = WORK / "official_evaluator" / k
    shutil.rmtree(wd, ignore_errors=True); (wd / "teams" / "P04").mkdir(parents=True)
    try:
        download(EVALUATOR[0], wd / EVALUATOR[1])
    except Exception as e:
        out["reason"] = f"the evaluator did not download ({e.__class__.__name__})"
        return out
    got = sha256(wd / EVALUATOR[1])
    if got != EVALUATOR[2]:
        out["reason"] = f"the evaluator is not the recorded file (its SHA-256 starts {got[:16]})"
        return out
    try:
        _, kept, types_in_file = read_labels()
    except (SystemExit, Exception) as e:
        out["reason"] = f"the label file is not the recorded one, or it did not download ({str(e)[:120]})"
        return out
    # the evaluator stops on a machine type without score files (the four MIMII types are not scored here), so it gets the label
    # file's sections of the two toy types only, with their rows unchanged
    with open(wd / "eval_data_list.csv", "w", newline="") as fh:
        fh.writelines(kept)
    for f in files:
        shutil.copy(src / f, wd / "teams" / "P04" / f)
    r = subprocess.run([sys.executable, EVALUATOR[1]], cwd=wd, capture_output=True, text=True)   # the official script, unmodified
    res_f = wd / "teams" / "result_P04.csv"
    if r.returncode != 0 or not res_f.exists():
        out["reason"] = f"the evaluator stopped (code {r.returncode}): {(r.stderr or r.stdout)[-300:]}"
        return out
    off, cur = {}, None
    with open(res_f, newline="") as fh:
        for row in csv.reader(fh):
            if len(row) == 1:
                cur = row[0]; off[cur] = {"per_machine": {}}
            elif len(row) == 3 and row[0] == "Average":
                off[cur]["AUC"], off[cur]["pAUC"] = float(row[1]), float(row[2])
            elif len(row) == 3 and row[0] != "id":
                off[cur]["per_machine"]["id_" + row[0]] = {"AUC": float(row[1]), "pAUC": float(row[2])}
    ours = json.loads(EV_RES.read_text())["systems"][k]["per_type"]
    if set(off) != set(TYPES) or any(set(off[t]["per_machine"]) != set(ours[t]["per_machine"]) or "AUC" not in off[t] for t in TYPES):
        out["reason"] = "the evaluator's result file does not list the same machine types and machines"
        return out
    diffs = [abs(off[t][m] - ours[t][m]) for t in TYPES for m in ("AUC", "pAUC")]
    diffs += [abs(off[t]["per_machine"][q][m] - ours[t]["per_machine"][q][m]) for t in TYPES for q in ours[t]["per_machine"] for m in ("AUC", "pAUC")]
    out.update(ran=True, agree=bool(max(diffs) <= EVAL_TOL), max_abs_difference=float(max(diffs)), values_compared=len(diffs),
               machine_types_in_the_label_file=types_in_file, label_rows_given=len(kept), per_type=off,
               this_notebook={t: {"AUC": ours[t]["AUC"], "pAUC": ours[t]["pAUC"]} for t in TYPES})
    shutil.copy(res_f, src / "official_evaluator_result.csv")
    return out


if EV_RES.exists():
    EVAL = json.loads(EV_RES.read_text())
    print(f"the evaluation set was scored on {EVAL['scores_written']}; its saved results are read, and nothing is scored again")
else:
    if not EV_LOCK.exists():
        files = {k: [] for k in EV_SETS}
        for k in EV_SETS:
            (EV_DIR / k).mkdir(exist_ok=True)
        for t in TYPES:
            for q in machines(t, "eval"):
                te = dc_rows(t, "eval", "test", q); names = [DMETA[t]["name"][i] for i in te]
                for k, nm in EV_SETS.items():
                    sc = score_of(nm, t, "eval", q, SEEDS)
                    assert len(sc) == len(te) and np.isfinite(sc).all(), (k, t, q)
                    f = EV_DIR / k / f"anomaly_score_{t}_{q}.csv"
                    f.write_text("".join(f"{a},{float(b)!r}\n" for a, b in sorted(zip(names, sc))))
                    files[k].append(f.name)
            FM.clear(); VM.clear()
        EV_LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "files": files, "test_run": TEST, "result_is": "kept",
                                       "systems": {"kept": {"form": "ZP", "view_seeds": SEEDS, "frozen": [[e, sd] for e, _, sd in FROZEN], "calibration": CAL},
                                                   "frozen": {"frozen": [[e, sd] for e, _, sd in FROZEN]},
                                                   "view": {"score": "the paper's score", "view_seeds": SEEDS}}}))
        print(f"the evaluation scores of the kept system and of its two parts are written ({sum(len(v) for v in files.values())} files), "
              "before any label is read", flush=True)
    lock = json.loads(EV_LOCK.read_text())
    GT, _lines, _ = read_labels()                                     # the labels: downloaded and read only here
    with open(EV_LABELS, "w", newline="") as fh:                      # the label rows of the two types, kept beside the scores
        fh.writelines(_lines)
    _sys = {}
    for k in EV_SETS:
        per = {}
        for fn in lock["files"][k]:
            t, q = re.match(r"anomaly_score_(\w+?)_(id_\d+)\.csv$", fn).groups()
            rows = [ln.split(",") for ln in (EV_DIR / k / fn).read_text().splitlines() if ln]
            missing = [a for a, _ in rows if (t, a) not in GT]
            assert not missing, f"no label for {missing[:3]}"
            per.setdefault(t, {})[q] = dc_metrics(np.array([GT[(t, a)] for a, _ in rows]), np.array([float(b) for _, b in rows]))
        _sys[k] = {"system": EV_SETS[k], "per_type": {t: {"per_machine": per[t], "AUC": float(np.mean([v["AUC"] for v in per[t].values()])),
                                                         "pAUC": float(np.mean([v["pAUC"] for v in per[t].values()]))} for t in TYPES}}
    EVAL = {"scored": True, "scores_written": lock["written"], "results_written": time.strftime("%Y-%m-%d %H:%M"), "result_is": "kept",
            "test_run": TEST, "ground_truth": list(DC_GT[1:]), "written_systems": lock["systems"], "systems": _sys}
    EV_RES.write_text(json.dumps(EVAL, indent=1))
    del GT

OFFICIAL = {}
for k in EV_SETS:
    try:
        OFFICIAL[k] = official_check(k)
    except Exception as e:                                            # this notebook's numbers do not depend on this check
        OFFICIAL[k] = {"ran": False, "reason": f"an unexpected error ({repr(e)[:200]})"}
(EV_DIR / "official_evaluator_check.json").write_text(json.dumps(OFFICIAL, indent=1))


def final_numbers(k, official=None):
    # the rules: if the official evaluator's numbers differ from this notebook's, the evaluator's are the result
    o = OFFICIAL[k] if official is None else official
    if o.get("ran") and not o.get("agree"):
        return {t: {"AUC": o["per_type"][t]["AUC"], "pAUC": o["per_type"][t]["pAUC"], "per_machine": o["per_type"][t]["per_machine"]} for t in TYPES}, \
            "the official evaluator (its numbers differ from this notebook's)"
    return ({t: EVAL["systems"][k]["per_type"][t] for t in TYPES},
            "this notebook (the official evaluator gives the same numbers)" if o.get("ran") else "this notebook (the official evaluator was not run)")


FINAL, FINAL_SOURCE = {}, {}
for k in EV_SETS:
    FINAL[k], FINAL_SOURCE[k] = final_numbers(k)
    o = OFFICIAL[k]
    print(f"the official evaluator on the {k} scores: " + (f"the same numbers: {o['agree']} (largest difference {o['max_abs_difference']:.1e} over "
          f"{o['values_compared']} values)" if o["ran"] else f"NOT run: {o['reason']}"))
    if o["ran"] and not o["agree"]:
        print("  the numbers differ: by the rules, the official evaluator's are the result:", o["per_type"])


def q1(t, auc, pauc):
    # the rules' word for a type: "with the teams" (above at least 3 of the 5), "above the baseline", "below the baseline"; AUCs in % of
    # two decimals, as the results page prints them. pAUC is reported against the same rows, with no word
    a, p = round(100 * auc, 2), round(100 * pauc, 2)
    above = sorted(k_ for k_, v in TEAMS.items() if a > v[t][0])
    word = "with the teams" if len(above) >= Q1_TEAMS else "above the baseline" if a >= BASELINE_EVAL[t][0] else "below the baseline"
    return {"AUC_percent": a, "pAUC_percent": p, "teams_below_on_AUC": len(above), "which": above, "baseline_AUC": BASELINE_EVAL[t][0], "word": word,
            "teams_below_on_pAUC": int(sum(p > v[t][1] for v in TEAMS.values())), "pAUC_at_least_the_baseline": bool(p >= BASELINE_EVAL[t][1]),
            "mean_of_AUC_and_pAUC": round((a + p) / 2, 2), "anopatch_mean_of_AUC_and_pAUC": ANOPATCH["evaluation"][t]}


def six_types(kept):
    # question 3: the six types together. The four MIMII types are notebook 25's evaluation results of the kept system; T3′ and the rank
    # among six systems (this one and the program's five teams) are the program's summaries, not an official rank
    ours = {**{t: N25_EVAL[t] for t in SIX if t not in TYPES}, **{t: (kept[t]["AUC"], kept[t]["pAUC"]) for t in TYPES}}
    table = {"this system": {t: (round(100 * ours[t][0], 2), round(100 * ours[t][1], 2)) for t in SIX}, **{k_: {t: tuple(v[t]) for t in SIX} for k_, v in TEAMS.items()}}
    ranks = {s_: {} for s_ in table}
    for t in SIX:
        for m, mn in enumerate(("AUC", "pAUC")):
            for s_ in table:                                          # rank 1 is the highest value; a tie shares the better rank
                ranks[s_][f"{t}|{mn}"] = 1 + int(sum(table[o_][t][m] > table[s_][t][m] for o_ in table))
    m6 = round(float(np.mean([ours[t][0] for t in SIX])), 4)
    return {"values_percent": table, "ranks": ranks, "mean_rank": {s_: round(float(np.mean(list(ranks[s_].values()))), 2) for s_ in table},
            "mean_AUC": {s_: round(float(np.mean([table[s_][t][0] for t in SIX])) / 100, 4) for s_ in table},
            "T3_prime": {"mean_AUC": m6, "mean_pAUC": round(float(np.mean([ours[t][1] for t in SIX])), 4), "bar": T3P_BAR, "reached": bool(m6 >= T3P_BAR),
                         "toy_types_mean_AUC": round(float(np.mean([ours[t][0] for t in TYPES])), 4),
                         "four_MIMII_types_mean_AUC": round(float(np.mean([ours[t][0] for t in SIX if t not in TYPES])), 4)}}


K = FINAL["kept"]
Q1 = {t: q1(t, K[t]["AUC"], K[t]["pAUC"]) for t in TYPES}
Q2 = fusion_table({nm: {t: FINAL[k][t] for t in TYPES} for k, nm in EV_SETS.items()})
Q3 = six_types(K)
print(f"\nEvaluation set, scored once ({EVAL['scores_written']}) · AUC / pAUC · the result is the kept system's:")
for t in TYPES:
    print(f"  {NAME[t]:<12} kept system {K[t]['AUC']:.4f} / {K[t]['pAUC']:.4f} · frozen recipe {FINAL['frozen'][t]['AUC']:.4f} / {FINAL['frozen'][t]['pAUC']:.4f} · "
          f"view {FINAL['view'][t]['AUC']:.4f} / {FINAL['view'][t]['pAUC']:.4f}")
    print(f"  {'':<12} per machine (kept system): " + " ".join(f"{q} {v['AUC']:.4f}" for q, v in K[t]["per_machine"].items()))
print("\nQuestion 1 · the kept system beside the five teams and the baseline:")
for t in TYPES:
    print(f"  {NAME[t]:<12} AUC {Q1[t]['AUC_percent']:.2f} % → {Q1[t]['word'].upper()} (above {Q1[t]['teams_below_on_AUC']} of the 5 teams; baseline "
          f"{BASELINE_EVAL[t][0]:.2f}; the teams " + ", ".join(f"{v[t][0]:.2f}" for v in TEAMS.values()) + f") · pAUC {Q1[t]['pAUC_percent']:.2f} % (above "
          f"{Q1[t]['teams_below_on_pAUC']} of the 5; baseline {BASELINE_EVAL[t][1]:.2f}) · mean of AUC and pAUC {Q1[t]['mean_of_AUC_and_pAUC']:.2f} "
          f"(AnoPatch {ANOPATCH['evaluation'][t]:.2f})")
print("Question 2 · the fusion (the kept system less the better of its two parts):",
      {NAME[t]: (Q2[t]["kept_less_better_part"], Q2[t]["word"]) for t in TYPES}, "· on the development set:",
      {NAME[t]: (FUSION_DEV[t]["kept_less_better_part"], FUSION_DEV[t]["word"]) for t in TYPES})
_t3 = Q3["T3_prime"]
print(f"Question 3 · T3′: the mean AUC over the six types is {_t3['mean_AUC']} against {T3P_BAR}: reached {_t3['reached']} (the two toy types "
      f"average {_t3['toy_types_mean_AUC']}; the four MIMII types, notebook 25, {_t3['four_MIMII_types_mean_AUC']}) · mean pAUC {_t3['mean_pAUC']}")
print("  the mean rank among the six systems over the 12 columns (1 is best):", Q3["mean_rank"])
print("  this system's rank per column:", Q3["ranks"]["this system"])

In [ ]:
# 17 · Save, one picture, and one zip to download
import matplotlib.pyplot as plt

RESULT = {"rules": RULES, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "gpu_gb": round(GPU_GB, 1), "torch": torch.__version__, "torchaudio": torchaudio.__version__,
                   "clip_samples_kept": int(NS),
                   "frozen_recipe": {"cfg": CFG, "system": [[e, c, sd] for e, c, sd in FROZEN], "classes": F_CLASSES, "n_train": F_TRAIN,
                                     "encoders": ENC_INFO, "GB": GBR, "echo": INFO["ECHO"], "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                                     "train_and_pass_seconds": T_TRAIN_F, "echo_seconds": T_ECHO, "part_1_seconds": T_PART1},
                   "view": {"reference": {"repository": "https://github.com/soonhyeon/Noisy-ArcMix", "commit": REF_COMMIT, "licence": "MIT, Copyright (c) 2023 soonhyeon"},
                            "sv": SV, "n_frames": int(N_FRAMES), "last_kernel": list(LAST_KERNEL), "classes": CLASSES, "n_clips": int(len(ITEMS)),
                            "memory_layout_seconds_per_step": MF_TIMES, "train_seconds": T_TRAIN, "embed_seconds": T_EMBED, "part_2_seconds": T_PART2},
                   "zips": {"development": {k: list(v[1:]) for k, v in DC_DEV.items()}, "additional": {k: list(v[1:]) for k, v in XZIPS.items()},
                            "evaluation": {k: list(v[1:]) for k, v in DC_EVAL.items()}, "ground_truth": list(DC_GT[1:])},
                   "prep_seconds": PREP_S, "session_minutes": round((time.time() - T_START) / 60, 1), "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": {**G0, "c": G0C}, "G1": {"frozen_recipe": {"ok": DC_G1, "losses": G1_F}, "view": G1}, "G_DEV": {"ok": GDEV_OK, "systems": GDEV, "bars": GDEV_BAR},
          "counts": DCASE["counts"], "clip_lengths_before_the_cut": DCASE["lengths"],
          "development": {"systems": DEV, "per_seed": PER_SEED, "fusion": FUSION_DEV},
          "calibration": CAL, "evaluation": EVAL, "official_evaluator": OFFICIAL,
          "final": {"numbers": FINAL, "source": FINAL_SOURCE, "result_is": "kept"},
          "readings": {"question_1": Q1, "question_2": {"evaluation": Q2, "development": FUSION_DEV}, "question_3": Q3},
          "yardsticks": {"teams_evaluation": TEAMS, "baseline_evaluation": BASELINE_EVAL, "published_development": PUBLISHED_DEV, "anopatch": ANOPATCH,
                         "notebook_25_evaluation_four_types": N25_EVAL, "T3_prime_bar": T3P_BAR},
          "training_logs": {"frozen_recipe": LOGS_F, "view": LOGS}}
(OUT / "toy_types.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
# every development test clip's score under every system, so that each development number above can be recomputed from the zip
SC_DIR = OUT / "development_scores"; SC_DIR.mkdir(exist_ok=True)
for (t, q), d in DEV_SC.items():
    (SC_DIR / f"scores_{t}_{q}.csv").write_text("file,label," + ",".join(SYSTEMS) + "\n" + "".join(
        f"{d['file'][i]},{int(d['label'][i])}," + ",".join(repr(float(d[nm][i])) for nm in SYSTEMS) + "\n" for i in range(len(d["file"]))))
# the clips the two spreads rest on: the frozen recipe's score of the additional-training clips it did not train on, and the paper's
# score of the view's held-out clips. No test clip.
CAL_DIR = OUT / "calibration_scores"; CAL_DIR.mkdir(exist_ok=True)
for t in TYPES:
    (CAL_DIR / f"frozen_{t}.csv").write_text("file,frozen_left_out\n" + "".join(f"{a},{float(b)!r}\n" for a, b in CALF_SC[t]))
    _rows = np.asarray(HOLD_ROWS[t], dtype=np.int64)
    (CAL_DIR / f"held_out_{t}.csv").write_text("file,paper\n" + "".join(
        f"{DMETA[t]['name'][int(r)]},{float(a)!r}\n" for r, a in zip(_rows, paper(SEEDS, t, _rows))))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
COL = {"frozen": "#2a78d6", "view": "#eb6834", "fused": "#1baf7a"}    # three categorical slots, checked with the palette validator
MARK = {"frozen": "o", "view": "s", "fused": "D"}                    # a shape per system as well as a colour


def dots(ax, xs, vals, k, lab):
    # every mark carries its value, on a white patch so that a line behind it does not cross the digits
    ax.plot(xs, vals, MARK[k], ms=9, color=COL[k], mec="white", mew=1.2, label=lab, zorder=3)
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    for a, v in zip(xs, vals):
        ax.text(a, v + 0.03 * span, f"{v:.4f}", ha="center", va="bottom", fontsize=8, color=INK, zorder=4,
                bbox=dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.9))


fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.4))
x, w = np.arange(len(TYPES)), 0.24
panels = [(axes[0], "Development set (read once; it selected nothing)", {k: [DEV[nm][t]["AUC"] for t in TYPES] for k, nm in EV_SETS.items()}),
          (axes[1], f"Evaluation set (scored once, {EVAL['scores_written']})", {k: [FINAL[k][t]["AUC"] for t in TYPES] for k in EV_SETS})]
for ax, ttl, v in panels:
    ev = ax is axes[1]
    ticks = ([[tm[t][0] / 100 for tm in TEAMS.values()] for t in TYPES] if ev
             else [[PUBLISHED_DEV["Noisy-ArcMix, TASTgram-MFN (arXiv 2310.06364, Table 1)"][t][0] / 100] for t in TYPES])
    base = [BASELINE_EVAL[t][0] / 100 for t in TYPES] if ev else [PUBLISHED_DEV["DCASE 2020 baseline (the task page)"][t][0] / 100 for t in TYPES]
    lo = math.floor((min([a for vals in v.values() for a in vals] + [a for tk in ticks for a in tk] + base) - 0.03) * 20) / 20
    ax.set_ylim(lo, 1.0 + 0.1 * (1.0 - lo))
    dots(ax, x - w, v["frozen"], "frozen", "frozen recipe alone")
    dots(ax, x, v["view"], "view", "second view alone (the paper's score)")
    dots(ax, x + w, v["kept"], "fused", "kept system: frozen + view, form ZP" + (" (the result)" if ev else ""))
    for i, tk in enumerate(ticks):                                    # the yardsticks: short ink ticks at the right of each type's group
        ax.hlines(tk, i + 0.36, i + 0.47, color=INK, lw=1.3, zorder=2,
                  label=("the program's five 2020 teams" if ev else "Noisy-ArcMix as published (six types together)") if i == 0 else None)
    ax.hlines(base, x - 0.47, x + 0.47, color=MUTED, lw=1.1, ls=(0, (4, 2)), zorder=1, label="DCASE 2020 baseline")
    ax.set_title(ttl, loc="left", color=INK, fontsize=10.5)
    ax.set_xticks(x, [NAME[t] for t in TYPES]); ax.set_xlim(-0.6, len(TYPES) - 0.4)
    ax.set_ylabel("AUC (mean over a type's machines); the scale does not start at zero", color=MUTED, fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.tick_params(colors=MUTED)
    ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
_w = " · ".join(f"{NAME[t]}: {Q1[t]['word']}" for t in TYPES)
fig.suptitle("P04 · notebook 27 · the kept system on DCASE 2020's ToyCar and ToyConveyor (two machine types no earlier notebook read)\n"
             f"evaluation set: {_w} · six-type mean AUC {Q3['T3_prime']['mean_AUC']} against {T3P_BAR}", fontsize=11, x=0.012, y=0.985, ha="left")
fig.subplots_adjust(left=0.065, right=0.985, top=0.84, bottom=0.2, wspace=0.17); fig.savefig(OUT / "toy_types.png", dpi=150); plt.show()

ZIP = OUT / "p04_n27_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("toy_types.json", "toy_types.png"):
        z.write(OUT / f, f)
    for d in (SC_DIR, CAL_DIR):
        for f in sorted(d.glob("*.csv")):
            z.write(f, f"{d.name}/{f.name}")
    for f in sorted(p for p in EV_DIR.rglob("*") if p.is_file()):
        z.write(f, f"{EV_DIR.name}/{f.relative_to(EV_DIR).as_posix()}")
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n27_results.zip`). It is also in Drive › `P04_ToyTypes`.